# Handiprint occlusion robustness: GPU reproduction

A separate reproduction of the earlier exploratory study. Original pair split, preprocessing, model settings, seeds and mask rules are retained. CNN training and inference require CUDA. Image processing, augmentation preparation, traditional baselines and statistics still use CPU. Prior test exposure remains a limitation. This does not replace the frozen course submission.


# Evaluating CNN Robustness to Partial Occlusion in Footwear Impression Classification
Dora Zhuang | Oct 9, 2026

## Project overview

Incomplete footwear impressions may contain less information for automated
classification. This project investigates how the location and extent of
simulated occlusion affect a CNN's ability to distinguish two footwear
patterns in the ShoeCase Handiprint dataset.

## Research question

**How does partial occlusion affect classification performance, and how
does a CNN compare with a traditional image-feature baseline?**

The dataset labels, Adidas and Nike, represent two specific shoe models.
This is a small proof-of-concept study using reference impressions, rather
than a system for identifying suspects or recognising footwear brands
in general. The simulated masks do not reproduce all the characteristics
of real crime-scene impressions.

## Data source citation

Tibben, Abigail; Carriquiry, Alicia; McGuire, Megan (2023). ShoeCase: A data set of mock crime scene footwear impressions. Iowa State University. Dataset. https://doi.org/10.25380/iastate.22814438.v1

## Analysis plan

1. Match image files to the metadata and document any exclusions.
2. Split the data into training, validation, and test sets by shoe pair.
3. Preprocess the images and inspect the cropped impressions.
4. Train a HOG-based logistic regression baseline and a pretrained CNN.
5. Compare performance on unmodified images and images with simulated occlusion.
6. Discuss uncertainty, potential shortcut cues, limitations, and ethical implications.

Left and right impressions from the same shoe pair are related. The shoe pair is therefore treated as the unit of independence for data splitting and statistical inference.
- All images from each pair remain in the same split to prevent information leakage. The validation set is used for model selection. The selected models are then evaluated on the held-out test set. Analyses introduced after reviewing test results are reported as exploratory.
- Bootstrap confidence intervals are calculated by resampling shoe pairs, keeping their images together. Results are reported at both the image level and the shoe-pair level.

## 0. Setup

Import the required packages and define the file paths, preprocessing settings, model settings, and evaluation plan. Random seeds and software versions are recorded to support reproducibility.

In [ ]:
# Dependencies are supplied by the GPU launcher.

# File handling and runtime information.
import os, sys, json, hashlib, re, math, time
import warnings, platform, zipfile, zlib
from pathlib import Path

# Arrays and metadata tables.
import numpy as np
import pandas as pd

# Plotting and image loading.
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

# Image processing and HOG features.
from scipy import ndimage as ndi
from skimage import filters, morphology, measure, feature

# Data splitting and the logistic regression baseline.
import sklearn
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Classification metrics.
from sklearn.metrics import log_loss, balanced_accuracy_score

# CNN training and pretrained models.
import torch
import torch.nn as nn
import torchvision

# Allow large TIFF scans.
Image.MAX_IMAGE_PIXELS = None

# Hide selected warnings.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Set figure resolution and disable grids.
plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.grid": False
})

## 0.1 Configuration

In [ ]:
# Read an environment setting; only "1" means True.
def _env_bool(name, default):
    v = os.environ.get(name)
    return default if v is None else v.strip() == "1"


# Synthetic-data test mode; keep disabled for the real experiment.
SMOKE = _env_bool("NOTEBOOK_SMOKE", False)


# ---- File paths ----
# Use default folders unless environment overrides are provided.
ROOT = Path(os.environ.get("PROJECT_ROOT", ".")).resolve()
XLSX = ROOT / "data" / "Handiprint_database.xlsx"
IMAGE_DIR = Path(
    os.environ.get("IMAGE_DIR", ROOT / "data" / "Handiprint_images")
)
RESULTS = Path(os.environ.get("RESULTS_DIR", ROOT / "results"))

# Create output folders if needed.
for sub in ["figures", "models", "cache", "tables"]:
    (RESULTS / sub).mkdir(parents=True, exist_ok=True)


# ---- Audit settings ----
ALLOW_EXCLUSIONS = True  # Exclude unresolved pairs and record the reasons.
NEAR_DUP_R = 0.98        # Similarity threshold for review.
NEAR_DUP_BLOCKS = False  # Flag near-duplicates without blocking the audit.


# ---- Data split ----
SEED = 42
VAL_FRAC, TEST_FRAC = 0.20, 0.20  # Approximately 60% remains for training.
MIN_PAIRS_PER_CLASS = 10


# ---- Preprocessing ----
# Apply the same rules to all images without using class labels.
CANVAS = 224                # Preserve aspect ratio with white padding.
SEG_TARGET_H = 650          # Approximate height used for segmentation.
INK_LO, INK_HI = 0.07, 0.20 # Include faint ink connected to stronger ink.
CLOSE_RADIUS = 12           # Connect nearby regions on the reduced image.
MIN_COMPONENT_FRAC = 0.003  # Ignore components below 0.3% of image area.
BORDER_FRAC = 0.02          # Ignore scanner edges.
PAD_FRAC = 0.03             # Add padding around the impression.
HOG_SIZE = 128              # Image size for HOG extraction.


# ---- Simulated occlusion ----
# Mask areas refer to the impression bounding box.
REGION_FRACTION = 1 / 3
RANDOM_LEVELS = (0.10, 0.25, 0.40)
K_MASKS = 5                   # Random masks per image at each level.
MASK_ASPECT_RANGE = (0.5, 2.0) # Mask width-to-height ratio.


# ---- Model settings ----
# Select hyperparameters using the validation set.
HOG_C_GRID = (0.01, 0.1, 1.0, 10.0)  # Larger C means weaker regularisation.
LR_GRID = (1e-3, 1e-2)               # Candidate CNN learning rates.
SEEDS = (0, 1, 2)                    # Repeat CNN training with three seeds.

MAX_EPOCHS = 40
PATIENCE = 7        # Stop after seven epochs without validation improvement.
BATCH = 16
WEIGHT_DECAY = 1e-4 # Optimizer regularisation.
PRETRAINED = True  # Start from pretrained weights.

TRY_FINETUNE = False  # Optional layer4 fine-tuning.
FT_MIN_VAL_LOSS_IMPROVEMENT = 0.05
RUN_ABSTENTION = False  # Optional rejection of uncertain predictions.


# ---- Confidence intervals ----
B_BOOT = 2000    # Shoe-pair bootstrap resamples.
CI = (2.5, 97.5) # Percentile limits for a 95% interval.


# Reduce computation for synthetic-data testing.
if SMOKE:
    MAX_EPOCHS = 3
    PATIENCE = 2
    B_BOOT = 50
    K_MASKS = 2
    SEEDS = (0, 1)
    LR_GRID = (1e-3,)
    PRETRAINED = False


# ---- Labels and random seeds ----
CLASSES = ["Adidas", "Nike"]  # Class 0 and class 1.

# Set seeds for the NumPy generator, NumPy global functions, and PyTorch.
rng_global = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ---- Computing device ----
# Use CUDA if available; otherwise use the CPU.
assert torch.cuda.is_available(), "CUDA is required."
assert not SMOKE, "Real-data reproduction requires SMOKE=False."
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
torch.set_num_threads(max(1, (os.cpu_count() or 2)))


# Save JSON, converting NumPy scalars to Python values.
def save_json(obj, path):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            default=lambda o: o.item() if hasattr(o, "item") else str(o)
        )
    )


# Hash files in chunks to limit memory use.
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


# Display software versions and active settings.
print(
    "python", platform.python_version(),
    "| torch", torch.__version__,
    "| torchvision", torchvision.__version__,
    "| numpy", np.__version__,
    "| pandas", pd.__version__,
    "| sklearn", sklearn.__version__
)
print("device:", DEVICE, "| SMOKE (synthetic) mode:", SMOKE)
print("image folder:", IMAGE_DIR)

## 1. Read the metadata

The `matched prints` sheet links each handiprint file to a shoe-pair ID, a side (L/R), and a brand label.

The `Image-info` sheet provides additional details, including shoe size, colour, and model. These details are matched by file name. Records without a match are retained and flagged. Each shoe pair is checked to ensure that it contains one left and one right impression with the same class label.

**Class labels are taken only from `matched prints`.** Surrounding spaces are removed from file names and brand labels. Brand labels are standardised to a consistent letter case.

In [ ]:
# Read the metadata sheets and inspect their columns.
with pd.ExcelFile(XLSX) as xl:
    print("sheets:", xl.sheet_names)
    raw_match = xl.parse("matched prints")
    raw_info = xl.parse("Image-info")

print("matched prints columns:", list(raw_match.columns))
print("Image-info columns:", list(raw_info.columns))


# Rename columns and preserve the original values.
meta = raw_match.rename(columns={
    "file name": "file_name_raw",
    "L/R": "side",
    "ID #": "pair_id",
    "adidas/nike": "brand_raw"
}).copy()

# Remove surrounding spaces and standardise brand labels.
meta["file_name"] = meta["file_name_raw"].str.strip()
meta["brand"] = meta["brand_raw"].str.strip().str.title()

# Check class labels and left/right values.
assert set(meta["brand"]) == set(CLASSES), (
    f"unexpected labels: {set(meta['brand'])}"
)
assert set(meta["side"]) == {"L", "R"}


# Extract file details; the .tif extension is optional.
pat = re.compile(
    r"^(\d{6})([LR])_(\d{8})_(\d+)_(\d+)_(\d+)_(.*?)(?:\.tif)?$"
)
parsed = meta["file_name"].str.extract(pat)
parsed.columns = [
    "file_number", "side_in_name", "date",
    "_a", "image_no", "replicate_no", "workers"
]
meta = pd.concat([meta, parsed.drop(columns="_a")], axis=1)

# Check the file-name pattern and left/right agreement.
assert meta["file_number"].notna().all(), (
    "some file names do not follow the documented pattern"
)
assert (meta["side"] == meta["side_in_name"]).all(), (
    "L/R column disagrees with the file name"
)

# Convert dates to numbers for sensitivity analyses.
meta["date_ordinal"] = (
    pd.to_datetime(meta["date"], format="%Y%m%d")
    .map(lambda d: d.toordinal())
)

# Encode Adidas as 0 and Nike as 1.
meta["y"] = meta["brand"].map({
    c: i for i, c in enumerate(CLASSES)
})


# Prepare additional shoe details.
info = raw_info.copy()
info["file_name"] = info["File Name"].str.strip()

# Stop if duplicate file names require review.
assert not info["file_name"].duplicated().any(), (
    "duplicate file names in Image-info; inspect before merging"
)

info = info[[
    "file_name", "Shoe Size", "Shoe Color", "Shoe Make/Model"
]].copy()
info.columns = [
    "file_name", "shoe_size", "shoe_color", "shoe_make_model"
]

# Keep primary records even when extra details are unavailable.
meta = meta.merge(
    info,
    on="file_name",
    how="left",
    validate="many_to_one",
    indicator="_info_match"
)

# Track matching records separately from missing shoe details.
meta["in_image_info"] = meta["_info_match"].eq("both")
meta = meta.drop(columns="_info_match")


# Summarise records by shoe pair.
pairs_meta = (
    meta.groupby("pair_id")
    .agg(
        brand=("brand", "first"),
        n_brand=("brand", "nunique"),
        n_rows=("file_name", "size"),
        sides=("side", lambda s: "".join(sorted(s)))
    )
    .reset_index()
)

# Require one left and one right record with the same class label.
assert (pairs_meta["n_brand"] == 1).all(), (
    "inconsistent class labels within a shoe pair"
)
assert (pairs_meta["sides"] == "LR").all(), (
    "each shoe pair must contain exactly one left and one right record"
)

# Summarise the metadata before checking image files.
print(
    f"\n{len(meta)} metadata rows | {len(pairs_meta)} shoe pairs "
    "| pairs per class:",
    pairs_meta["brand"].value_counts().to_dict()
)
print(
    "rows without a matching Image-info entry:",
    int((~meta["in_image_info"]).sum())
)

## 2. Audit the image folder (Gate 1)

Each metadata record is matched to an image file. Exact matches and unique formatting-only matches are accepted. Missing or ambiguous matches are flagged. Alternative versions are not selected automatically.

| Status | Meaning | Accepted match? |
|---|---|---|
| `EXACT` | One exact name match, with no competing formatting-only match. | Yes |
| `FORMAT_ONLY` | One match after ignoring surrounding spaces, letter case, and the TIFF extension. | Yes, recorded |
| `AMBIGUOUS_DUPLICATE_NAME` | Multiple files have the same normalised name. | No |
| `AMBIGUOUS_DIFFERENT_VERSION` | No name match, but other files share the same six-digit number and side. | No |
| `MISSING` | No matching file or alternative version. | No |

Additional checks flag unreadable files, ink in extra TIFF frames, and byte-identical files assigned to different shoe pairs or to both sides of one pair. Only the first frame (frame 0) is used for analysis.

Highly similar thumbnails are flagged for review but do not block the audit by default, because impressions from the same shoe model may look similar. A second check after preprocessing flags highly similar images across data splits.

Unresolved records stop the analysis unless exclusions are enabled. When exclusions are enabled, the entire affected shoe pair is excluded and the reason is recorded.

In [ ]:
# Check that the image folder exists.
if not IMAGE_DIR.exists():
    raise FileNotFoundError(
        f"{IMAGE_DIR} does not exist. Extract the handiprint TIFF files there."
    )

# Find TIFF files in the folder and its subfolders.
all_files = [p for p in IMAGE_DIR.rglob("*") if p.is_file()]
tif_paths = [
    p for p in all_files
    if p.suffix.lower() in (".tif", ".tiff")
]

if not tif_paths:
    raise FileNotFoundError(f"No TIFF files found under {IMAGE_DIR}.")

# Record file names and relative paths.
files = pd.DataFrame({
    "relpath": [p.relative_to(IMAGE_DIR).as_posix() for p in tif_paths],
    "name": [p.name for p in tif_paths]
})

# Normalise names for matching without renaming files.
files["fmt_key"] = (
    files["name"].str.strip().str.lower()
    .str.replace(r"\.tiff?$", "", regex=True)
)
files["num_side"] = files["name"].str[:7]

print(
    f"{len(all_files)} files in folder, "
    f"{len(files)} are .tif/.tiff, "
    f"{len(all_files) - len(files)} are other file types"
)


# Match one metadata record to candidate files.
def match_row(r):
    key = re.sub(r"\.tiff?$", "", r.file_name_raw.strip().lower())

    exact = files[files["name"] == r.file_name_raw]
    fmt = files[files["fmt_key"] == key]

    # Find alternative versions with the same file number and side.
    same_num = files[files["num_side"] == r.file_name[:7]]

    # Accept only a unique name match.
    if len(exact) == 1 and len(fmt) == 1:
        status, cand = "EXACT", exact
    elif len(exact) == 0 and len(fmt) == 1:
        status, cand = "FORMAT_ONLY", fmt
    elif len(fmt) > 1:
        status, cand = "AMBIGUOUS_DUPLICATE_NAME", fmt
    elif len(same_num) >= 1:
        status, cand = "AMBIGUOUS_DIFFERENT_VERSION", same_num
    else:
        status, cand = "MISSING", files.iloc[0:0]

    accepted = status in ("EXACT", "FORMAT_ONLY")
    used = cand["relpath"].iloc[0] if accepted else None

    # Record alternative versions without selecting them.
    others = sorted(
        set(same_num["relpath"]) - ({used} if used else set())
    )

    return pd.Series({
        "match_status": status,
        "matched_path": used,
        "candidate_paths": (
            ";".join(cand["relpath"])
            if status.startswith("AMBIG") else ""
        ),
        "alt_versions_in_folder": (
            ";".join(others) if accepted else ""
        )
    })


# Attach matching results to the metadata.
audit = pd.concat([meta, meta.apply(match_row, axis=1)], axis=1)
print(audit["match_status"].value_counts().to_string())


# List files without an accepted or ambiguous metadata match.
used_paths = set(audit["matched_path"].dropna())
referenced_ambig = (
    set(";".join(audit["candidate_paths"]).split(";")) - {""}
)
unref = files[
    ~files["relpath"].isin(used_paths | referenced_ambig)
]

# Save the inventory; these files are not used in the analysis.
unref.to_csv(
    RESULTS / "tables" / "audit_unreferenced_files.csv",
    index=False
)
print(
    f"{len(unref)} TIFF files have no accepted or ambiguous metadata match "
    "(listed in tables/audit_unreferenced_files.csv, not used)"
)

### 2.1 Inspect matched images

Read each matched file and record its hash, size, dimensions, and frame count. Estimate ink coverage in each frame and create a thumbnail for similarity checks. Scale 16-bit grayscale images to 8-bit before inspection. Files that cannot be read are flagged.

In [ ]:
# Convert the current frame to 8-bit grayscale.
def frame_to_gray_uint8(im):
    raw = np.asarray(im)
    if raw.ndim == 2 and (
        (raw.dtype.kind == "u" and raw.dtype.itemsize == 2)
        or im.mode.startswith("I;16")
    ):
        gray = np.round(raw.astype(np.float32) * (255.0 / 65535.0))
        return np.clip(gray, 0, 255).astype(np.uint8)

    return np.asarray(im.convert("L")).copy()


# Estimate ink coverage relative to paper brightness.
def ink_fraction_small(gray_small):
    s = gray_small.astype(np.float32) / 255.0
    bg = np.median(s)
    ink = np.clip((bg - s) / max(bg, 1e-3), 0, 1)
    return float((ink > 0.12).mean())


# Inspect all frames and create a thumbnail from frame 0.
def inspect_file(path):
    out = {"readable": False}

    try:
        out["sha256"] = sha256_file(path)
        out["bytes"] = path.stat().st_size

        with Image.open(path) as im:
            out.update(
                width=im.width,
                height=im.height,
                mode=im.mode,
                dpi=str(im.info.get("dpi")),
                n_frames=getattr(im, "n_frames", 1)
            )

            # Reduce frame 0 for ink and similarity checks.
            im.seek(0)
            g0 = Image.fromarray(frame_to_gray_uint8(im))
            small = np.asarray(g0.reduce(16))

            out["ink_frac_frame0"] = ink_fraction_small(small)
            out["thumb"] = np.asarray(
                Image.fromarray(small).resize(
                    (48, 96), Image.BILINEAR
                ),
                dtype=np.float32
            )

            # Check whether additional frames contain ink.
            other = []
            for k in range(1, out["n_frames"]):
                im.seek(k)
                gray = Image.fromarray(frame_to_gray_uint8(im))
                other.append(
                    ink_fraction_small(np.asarray(gray.reduce(16)))
                )

            out["ink_frac_other_frames_max"] = max(other) if other else 0.0
            out["readable"] = True

    except Exception as e:
        # Record failures for the audit.
        out["error"] = f"{type(e).__name__}: {e}"

    return out


# Inspect each matched file once.
t0 = time.time()
file_info = {}
matched_unique = sorted(used_paths)

for i, rel in enumerate(matched_unique, 1):
    file_info[rel] = inspect_file(IMAGE_DIR / rel)

    if i % 20 == 0 or i == len(matched_unique):
        print(
            f"  inspected {i}/{len(matched_unique)} files "
            f"({time.time() - t0:.0f}s)"
        )


# Keep thumbnails in memory and put other details in a table.
finfo = pd.DataFrame.from_dict(
    {
        path: {
            key: value
            for key, value in details.items()
            if key != "thumb"
        }
        for path, details in file_info.items()
    },
    orient="index"
)
finfo.index.name = "matched_path"

# Attach inspection results without dropping unmatched metadata rows.
audit = audit.merge(
    finfo.reset_index(),
    on="matched_path",
    how="left"
)

### 2.2 Check duplicates and TIFF frames

Flag matching problems, unreadable files, and ink in additional TIFFframes. File hashes identify byte-identical files. Thumbnail correlations flag similar images for review; similarity alone does not confirm duplication.

Blocking flags are summarised by shoe pair. Audit records, duplicate checks, and image summaries are saved for review.

In [ ]:
# Store audit flags separately for each record.
audit["flag_list"] = [[] for _ in range(len(audit))]


def add_flag(idx, flag):
    audit.at[idx, "flag_list"] = audit.at[idx, "flag_list"] + [flag]


# Flag matching problems, unreadable files, and additional TIFF frames.
for i, r in audit.iterrows():
    st = r.match_status

    if st.startswith("AMBIG") or st == "MISSING":
        add_flag(i, st)
    elif st == "FORMAT_ONLY":
        add_flag(i, "FORMAT_ONLY_NAME_DIFFERENCE")

    has_file = pd.notna(r.matched_path)
    readable = bool(r.readable) if pd.notna(r.readable) else False

    if has_file and not readable:
        add_flag(i, "UNREADABLE")

    if pd.notna(r.alt_versions_in_folder) and r.alt_versions_in_folder:
        add_flag(i, "ALT_VERSIONS_PRESENT(not used)")

    if readable:
        if r.n_frames > 1:
            add_flag(i, "MULTI_FRAME_TIFF(frame0 used)")

            # Flag extra frames with ink coverage above 20% of frame 0's.
            if r.ink_frac_other_frames_max > 0.2 * max(
                r.ink_frac_frame0, 1e-6
            ):
                add_flag(i, "INK_IN_OTHER_FRAMES")

        if r.ink_frac_frame0 < 0.005:
            add_flag(i, "VERY_LITTLE_INK_FRAME0")


# Identify byte-identical files using their hashes.
dups = []
ok = audit[audit["sha256"].notna()]

for h, grp in ok.groupby("sha256"):
    if len(grp) > 1:
        same_pair = grp["pair_id"].nunique() == 1

        dups.append({
            "kind": "byte_identical",
            "rows": ";".join(grp["file_name"]),
            "pair_ids": ";".join(
                map(str, sorted(grp["pair_id"].unique()))
            ),
            "same_pair": same_pair
        })

        flag = (
            "DUPLICATE_IDENTICAL_FILES_SAME_PAIR"
            if same_pair else
            "DUPLICATE_IDENTICAL_FILES_CROSS_PAIR"
        )
        for i in grp.index:
            add_flag(i, flag)


# Compare readable files using standardised grayscale thumbnails.
keys = [k for k in matched_unique if file_info[k]["readable"]]

if len(keys) > 1:
    T = np.stack([file_info[k]["thumb"].ravel() for k in keys])
    T = (
        (T - T.mean(axis=1, keepdims=True))
        / (T.std(axis=1, keepdims=True) + 1e-6)
    )
    R = (T @ T.T) / T.shape[1]

    # Link each file path to its metadata rows.
    path2rows = {}
    for idx, path in audit["matched_path"].items():
        if pd.notna(path):
            path2rows.setdefault(path, []).append(idx)

    # Compare each distinct file pair once.
    a_idx, b_idx = np.triu_indices(len(keys), k=1)
    similar = R[a_idx, b_idx] >= NEAR_DUP_R

    for a, b in zip(a_idx[similar], b_idx[similar]):
        ra = audit.loc[path2rows[keys[a]][0]]
        rb = audit.loc[path2rows[keys[b]][0]]
        same_pair = ra.pair_id == rb.pair_id

        dups.append({
            "kind": f"near_identical_r={R[a, b]:.3f}",
            "rows": f"{ra.file_name};{rb.file_name}",
            "pair_ids": f"{ra.pair_id};{rb.pair_id}",
            "same_pair": same_pair
        })

        flag = (
            "NEAR_DUPLICATE_REVIEW_SAME_PAIR"
            if same_pair else
            "NEAR_DUPLICATE_REVIEW_CROSS_PAIR"
        )
        for i in path2rows[keys[a]] + path2rows[keys[b]]:
            add_flag(i, flag)

pd.DataFrame(
    dups,
    columns=["kind", "rows", "pair_ids", "same_pair"]
).to_csv(
    RESULTS / "tables" / "audit_duplicates.csv",
    index=False
)


# Separate blocking problems from review-only flags.
BLOCKING = {
    "AMBIGUOUS_DUPLICATE_NAME",
    "AMBIGUOUS_DIFFERENT_VERSION",
    "MISSING",
    "UNREADABLE",
    "INK_IN_OTHER_FRAMES",
    "DUPLICATE_IDENTICAL_FILES_SAME_PAIR",
    "DUPLICATE_IDENTICAL_FILES_CROSS_PAIR"
}

if NEAR_DUP_BLOCKS:
    BLOCKING |= {
        "NEAR_DUPLICATE_REVIEW_SAME_PAIR",
        "NEAR_DUPLICATE_REVIEW_CROSS_PAIR"
    }

audit["blocking_flags"] = audit["flag_list"].map(
    lambda flags: [f for f in flags if f in BLOCKING]
)
audit["blocking"] = audit["blocking_flags"].map(len) > 0

# Convert flag lists to text for CSV output.
audit["flags_str"] = audit["flag_list"].map(";".join)
audit["blocking_str"] = audit["blocking_flags"].map(";".join)

audit.drop(columns=["flag_list", "blocking_flags"]).to_csv(
    RESULTS / "tables" / "audit_records.csv",
    index=False
)


# A pair passes only if both records have no blocking flags.
pair_status = (
    audit.groupby("pair_id")
    .agg(
        brand=("brand", "first"),
        n_rows=("blocking", "size"),
        n_blocking=("blocking", "sum"),
        reasons=(
            "blocking_str",
            lambda s: ";".join(sorted({
                flag
                for text in s
                for flag in text.split(";")
                if flag
            }))
        )
    )
    .reset_index()
)
pair_status["pair_ok"] = (
    (pair_status["n_blocking"] == 0)
    & (pair_status["n_rows"] == 2)
)


# Summarise readable images before pair exclusions.
inv = audit[audit["readable"].fillna(False).astype(bool)]

print(
    "\nImage formats/modes:", inv["mode"].value_counts().to_dict(),
    "| dpi:", inv["dpi"].value_counts().to_dict(),
    "| frames:", inv["n_frames"].value_counts().to_dict()
)
print(
    "Distinct (width x height) sizes:",
    inv.groupby(["width", "height"]).ngroups
)

inv.groupby("brand").agg(
    n=("width", "size"),
    width_median=("width", "median"),
    height_median=("height", "median"),
    bytes_median=("bytes", "median")
).to_csv(RESULTS / "tables" / "image_inventory_by_class.csv")


# Report flag counts and pairs requiring action.
flag_counts = pd.Series(
    [flag for flags in audit["flag_list"] for flag in flags],
    dtype=object
).value_counts()

print("\nFlag counts (metadata rows):")
print(flag_counts.to_string() if len(flag_counts) else "none")

print(
    "\nPairs OK:", int(pair_status["pair_ok"].sum()),
    "of", len(pair_status),
    "| pairs not passing:", int((~pair_status["pair_ok"]).sum())
)

if (~pair_status["pair_ok"]).any():
    print(
        pair_status.loc[
            ~pair_status["pair_ok"],
            ["pair_id", "brand", "reasons"]
        ].to_string(index=False)
    )


# Save the audit summary and pair-level decisions.
summary = {
    "metadata_rows": len(meta),
    "pairs": len(pair_status),
    "match_status": audit["match_status"].value_counts().to_dict(),
    "pairs_ok": int(pair_status["pair_ok"].sum()),
    "pairs_blocked": pair_status.loc[
        ~pair_status["pair_ok"], "pair_id"
    ].tolist(),
    "unreferenced_tifs": len(unref),
    "duplicate_groups": len(dups)
}

save_json(summary, RESULTS / "tables" / "audit_summary.json")
pair_status.to_csv(
    RESULTS / "tables" / "audit_pair_status.csv",
    index=False
)

## 3. Apply the audit gate and split by shoe pair

The notebook stops if unresolved pairs remain and exclusions are disabled. When exclusions are enabled, affected pairs are excluded and their reasons remain in the split manifest.

Usable pairs are split into approximately 60% training, 20% validation, and 20% test data, stratified by class. The split uses pair IDs and class labels, not image content. All images from each pair remain in one split.

The manifest is checked and saved with a file hash. Images are then preprocessed, but test predictions are made only after the experimental plan is locked.

In [ ]:
# Stop on unresolved pairs unless exclusions are enabled.
AUDIT_PASSED = bool(pair_status["pair_ok"].all())

if not AUDIT_PASSED and not ALLOW_EXCLUSIONS:
    raise RuntimeError(
        "IMAGE AUDIT DID NOT PASS.\n"
        f"{int((~pair_status['pair_ok']).sum())} pair(s) remain unresolved.\n"
        "Review tables/audit_pair_status.csv and tables/audit_records.csv. "
        "Fix the records or enable ALLOW_EXCLUSIONS."
    )

if not AUDIT_PASSED:
    print(
        "Excluded pairs (ALLOW_EXCLUSIONS=True):",
        pair_status.loc[~pair_status["pair_ok"], "pair_id"].tolist()
    )


# Check that both classes have enough usable pairs.
ok_pairs = pair_status.loc[
    pair_status["pair_ok"], ["pair_id", "brand"]
].reset_index(drop=True)

cls_counts = ok_pairs["brand"].value_counts()

if len(cls_counts) < 2 or cls_counts.min() < MIN_PAIRS_PER_CLASS:
    raise RuntimeError(
        "Too few usable pairs per class for the planned split: "
        f"{cls_counts.to_dict()}. Review the split design."
    )



# Reuse the original pair split; do not generate a new split.
ORIGINAL_MAN = ROOT / "data/split_manifest_final.csv"
EXPECTED_SPLIT_SHA = "20f942824a8c45c102837998cf2eb0b9f6d8cc9b243fa1ae87bef3db4f03d11d"
assert sha256_file(ORIGINAL_MAN) == EXPECTED_SPLIT_SHA

original_manifest = pd.read_csv(ORIGINAL_MAN)
original_used = original_manifest[
    original_manifest.split.isin(["train", "val", "test"])
].copy()

assert len(original_used) == 154
assert original_used.pair_id.nunique() == 77
assert set(original_manifest.loc[
    original_manifest.split == "EXCLUDED_UNRESOLVED", "pair_id"
]) == {81}
assert set(ok_pairs.pair_id) == set(original_used.pair_id), (
    "Audited pair cohort differs from the original cohort."
)

split_of = (
    original_used.drop_duplicates("pair_id")
    .set_index("pair_id")["split"].to_dict()
)

# Retain all metadata records, including excluded pairs.
man = audit[[
    "pair_id", "side", "brand", "y", "file_name",
    "match_status", "matched_path", "sha256", "date",
    "date_ordinal", "shoe_size", "flags_str"
]].copy()

man["split"] = (
    man["pair_id"].map(split_of).fillna("EXCLUDED_UNRESOLVED")
)

reasons = pair_status.set_index("pair_id")["reasons"]
man["exclusion_reason"] = (
    man["pair_id"].map(reasons)
    .where(man["split"] == "EXCLUDED_UNRESOLVED", "")
)
man = man.sort_values(
    ["split", "pair_id", "side"]
).reset_index(drop=True)


# Check pair integrity, file availability, and exact-content leakage.
def verify_manifest(m):
    used = m[m["split"] != "EXCLUDED_UNRESOLVED"]

    assert (used.groupby("pair_id")["split"].nunique() == 1).all(), (
        "a shoe pair appears in more than one split"
    )
    assert (
        used.groupby("pair_id")["side"]
        .apply(lambda s: "".join(sorted(s))) == "LR"
    ).all(), "a used pair must contain one left and one right record"

    assert not used.duplicated(["pair_id", "side"]).any(), (
        "duplicate (pair, side) rows"
    )

    # Identical file bytes must not appear across splits.
    assert used["sha256"].notna().all(), "a used file has no hash"
    assert (used.groupby("sha256")["split"].nunique() <= 1).all(), (
        "byte-identical files appear in multiple splits"
    )

    assert used["matched_path"].notna().all(), "a used file has no path"
    assert all(
        (IMAGE_DIR / p).exists() for p in used["matched_path"]
    ), "a used image path does not exist"

    assert (used.groupby("pair_id")["brand"].nunique() == 1).all(), (
        "inconsistent class labels within a shoe pair"
    )
    return True


print("manifest verified:", verify_manifest(man))



# Compare fresh included records with the original source identities.
columns = [
    "pair_id", "side", "brand", "y", "file_name",
    "matched_path", "sha256", "split"
]
actual = (
    man[man.split.isin(["train", "val", "test"])][columns]
    .sort_values(["pair_id", "side"]).reset_index(drop=True)
)
expected = (
    original_used[columns]
    .sort_values(["pair_id", "side"]).reset_index(drop=True)
)
pd.testing.assert_frame_equal(actual, expected, check_dtype=False)

MAN_PATH = ORIGINAL_MAN
MAN_SHA = sha256_file(MAN_PATH)
assert MAN_SHA == EXPECTED_SPLIT_SHA

(RESULTS / "original_split_manifest.csv").write_bytes(
    MAN_PATH.read_bytes()
)
man.to_csv(
    RESULTS / "tables/audited_manifest_gpu.csv", index=False
)
(RESULTS / "split_manifest_sha256.txt").write_text(MAN_SHA + "\n")
print("Original split verified; canonical manifest was not rewritten.")

# Compare with the provisional split, if available.
prov = ROOT / "data" / "split_manifest_provisional.csv"

if prov.exists():
    pv = (
        pd.read_csv(prov)[["pair_id", "split"]]
        .drop_duplicates()
        .set_index("pair_id")["split"]
    )
    fin = (
        man[man["split"] != "EXCLUDED_UNRESOLVED"]
        .drop_duplicates("pair_id")
        .set_index("pair_id")["split"]
    )
    common = fin.index.intersection(pv.index)

    print(
        "final vs provisional split:",
        int((fin.loc[common] != pv.loc[common]).sum()),
        "of", len(common), "pairs differ"
    )


# Report shoe-pair counts separately from image counts.
included = man[man["split"] != "EXCLUDED_UNRESOLVED"]

n_tab = (
    included.drop_duplicates("pair_id")
    .groupby(["split", "brand"])
    .size()
    .unstack(fill_value=0)
)
img_tab = (
    included.groupby(["split", "brand"])
    .size()
    .unstack(fill_value=0)
)

print("\nSHOE PAIRS per split/class:\n", n_tab.to_string())
print("\nIMAGES per split/class (not independent):\n", img_tab.to_string())

n_tab.to_csv(RESULTS / "tables" / "split_pairs_per_class.csv")
img_tab.to_csv(RESULTS / "tables" / "split_images_per_class.csv")


# Prepare included records for preprocessing; test evaluation comes later.
use = included.reset_index(drop=True).copy()
use["key"] = use["file_name"]

## 4. Preprocess images

Apply the same preprocessing rules to every image. Settings are fixed, and brightness and contrast are calculated within each image; no statistics are pooled across splits.

1. Read frame 0 and convert it to 8-bit grayscale, scaling 16-bit TIFFs correctly.
2. Detect ink on a reduced image using relative darkness and hysteresis thresholding. Ignore the outer margin and small components.
3. Connect nearby tread regions to estimate a bounding box, then crop the original grayscale image with padding.
4. Normalise paper brightness and contrast, resize the longer side to 224 pixels, and centre the crop on a white square canvas.

The segmentation mask defines the crop; it does not erase pixels within it. Resizing standardises image framing but does not guarantee removal of shoe-size cues. Shape, ink coverage, scan texture, and sole logos may remain. These potential shortcuts are examined in the sensitivity analyses.

Processed images are cached for reuse. Failures stop the analysis rather than being dropped silently. A similarity check flags potential duplicates across splits for review.

In [ ]:
# Read frame 0 using the grayscale conversion defined in Section 2.1.
def read_gray_uint8(path):
    with Image.open(path) as im:
        im.seek(0)
        return frame_to_gray_uint8(im)


# Support both old and new scikit-image versions.
def remove_small(m, n):
    """Remove connected components of at most n pixels."""
    try:
        return morphology.remove_small_objects(m, max_size=n)
    except TypeError:
        return morphology.remove_small_objects(m, min_size=n + 1)


def segment_impression(gray_full):
    """Locate the impression and return its bounding box and QC measures."""
    H, W = gray_full.shape

    # Detect ink on a smaller image to reduce computation.
    f = max(1, int(round(H / SEG_TARGET_H)))
    s = (
        np.asarray(Image.fromarray(gray_full).reduce(f))
        .astype(np.float32) / 255.0
    )
    bg = float(np.median(s))
    ink = np.clip((bg - s) / max(bg, 1e-3), 0, 1)

    # Ignore outer edges that may contain scanner lines.
    h, w = s.shape
    b = max(1, int(round(BORDER_FRAC * min(h, w))))
    border = np.zeros_like(ink, dtype=bool)
    border[:b] = border[-b:] = True
    border[:, :b] = border[:, -b:] = True

    border_ink_frac = float(
        ((ink > INK_HI) & border).sum()
        / max((ink > INK_HI).sum(), 1)
    )
    ink[border] = 0

    # Keep faint ink connected to strong ink, then remove tiny components.
    mask = filters.apply_hysteresis_threshold(ink, INK_LO, INK_HI)
    mask = remove_small(mask, 15)

    # Connect nearby regions to estimate the crop boundary.
    closed = ndi.binary_closing(
        mask, structure=morphology.disk(CLOSE_RADIUS)
    )
    props = measure.regionprops(measure.label(closed))
    keep = [
        p for p in props
        if p.area >= MIN_COMPONENT_FRAC * h * w
    ]

    if not keep:
        raise ValueError("no impression found")

    y0 = min(p.bbox[0] for p in keep)
    x0 = min(p.bbox[1] for p in keep)
    y1 = max(p.bbox[2] for p in keep)
    x1 = max(p.bbox[3] for p in keep)

    # Record shape descriptors for QC; do not rotate the image.
    sub = mask[y0:y1, x0:x1]
    hh = sub.shape[0]
    ext = np.array([
        np.where(row)[0].max() - np.where(row)[0].min() + 1
        if row.any() else 0
        for row in sub
    ], dtype=float)

    third = max(1, int(round(0.3 * hh)))
    width_ratio_top_bottom = float(
        ext[:third].mean() / max(ext[-third:].mean(), 1e-6)
    )

    def cx(rows):
        _, xx = np.nonzero(rows)
        return xx.mean() if len(xx) else np.nan

    cx_top = cx(sub[:third])
    cx_mid = cx(sub[
        hh // 2 - third // 2:
        hh // 2 + third // 2 + 1
    ])
    cx_bot = cx(sub[-third:])
    waist_shift = float(
        (cx_mid - 0.5 * (cx_top + cx_bot))
        / max(x1 - x0, 1)
    )

    # Estimate background variation outside detected regions.
    bgmask = ~ndi.binary_dilation(closed, iterations=3)
    bgmask[border] = False

    info = {
        "raw_h": H,
        "raw_w": W,
        "paper_level": bg,
        "bg_std": float(s[bgmask].std()) if bgmask.any() else np.nan,
        "border_ink_frac": border_ink_frac,
        "n_components_total": len(props),
        "n_components_kept": len(keep),
        "ink_frac_in_box": float(sub.mean()),
        "width_ratio_top_bottom": width_ratio_top_bottom,
        "waist_shift": waist_shift,
        "seg_reduce": f
    }

    # Map the bounding box back to full-resolution coordinates.
    bbox = (
        y0 * f, x0 * f,
        min(y1 * f, H), min(x1 * f, W)
    )
    return bbox, info


def to_canvas(gray_full, bbox, paper_level):
    """Crop, normalise, and resize the impression onto a square canvas."""
    H, W = gray_full.shape
    y0, x0, y1, x1 = bbox

    # Add padding without extending beyond the image.
    py = int(round(PAD_FRAC * (y1 - y0)))
    px = int(round(PAD_FRAC * (x1 - x0)))
    y0, x0 = max(0, y0 - py), max(0, x0 - px)
    y1, x1 = min(H, y1 + py), min(W, x1 + px)

    # Normalise paper brightness and stretch contrast within the image.
    crop = gray_full[y0:y1, x0:x1].astype(np.float32) / 255.0
    x = np.clip(crop / max(paper_level, 1e-3), 0, 1)
    lo = np.percentile(x, 0.5)
    x = np.clip((x - lo) / max(1 - lo, 1e-3), 0, 1)

    # Preserve aspect ratio when resizing.
    h, w = x.shape
    sc = CANVAS / max(h, w)
    nh = max(1, int(round(h * sc)))
    nw = max(1, int(round(w * sc)))
    xr = np.asarray(
        Image.fromarray(x).resize((nw, nh), Image.LANCZOS)
    ).clip(0, 1)

    # Centre the crop on a white background.
    canvas = np.ones((CANVAS, CANVAS), dtype=np.float32)
    oy, ox = (CANVAS - nh) // 2, (CANVAS - nw) // 2
    canvas[oy:oy + nh, ox:ox + nw] = xr

    return (
        np.round(canvas * 255).astype(np.uint8),
        (oy, ox, oy + nh, ox + nw),
        (y0, x0, y1, x1)
    )


# Include the reader version so old 16-bit caches are not reused.
PREP_CFG = {
    "CANVAS": CANVAS,
    "SEG_TARGET_H": SEG_TARGET_H,
    "INK_LO": INK_LO,
    "INK_HI": INK_HI,
    "CLOSE_RADIUS": CLOSE_RADIUS,
    "MIN_COMPONENT_FRAC": MIN_COMPONENT_FRAC,
    "BORDER_FRAC": BORDER_FRAC,
    "PAD_FRAC": PAD_FRAC,
    "image_reader_version": "uint16_fixed_v1"
}

prep_hash = hashlib.sha256(
    json.dumps(PREP_CFG, sort_keys=True).encode()
).hexdigest()[:12]

CACHE_NPZ = RESULTS / "cache" / f"canvases_{prep_hash}.npz"
CACHE_TAB = RESULTS / "cache" / f"prep_table_{prep_hash}.csv"


# Reuse cached images when records cover the current dataset.
cached_prep = (
    pd.read_csv(CACHE_TAB)
    if CACHE_TAB.exists() else None
)
cache_ready = (
    CACHE_NPZ.exists()
    and cached_prep is not None
    and set(cached_prep["key"]) >= set(use["key"])
)

if cache_ready:
    with np.load(CACHE_NPZ) as z:
        cache_ready = set(z.files) >= set(use["key"])
        if cache_ready:
            canv = {k: z[k] for k in use["key"]}

if cache_ready:
    prep = cached_prep[
        cached_prep["key"].isin(use["key"])
    ].copy()
    print("loaded preprocessing cache", CACHE_NPZ.name)

else:
    canv, rows, t0 = {}, [], time.time()

    # Process each image and record failures without dropping rows.
    for i, r in use.iterrows():
        try:
            g = read_gray_uint8(IMAGE_DIR / r.matched_path)
            bbox, inf = segment_impression(g)
            cv, cbox, padded = to_canvas(g, bbox, inf["paper_level"])
            canv[r.key] = cv

            inf.update(
                key=r.key,
                bbox_h_px=bbox[2] - bbox[0],
                bbox_w_px=bbox[3] - bbox[1],
                bbox_aspect_w_over_h=(
                    (bbox[3] - bbox[1]) / max(bbox[2] - bbox[0], 1)
                ),
                bbox_y0=bbox[0],
                bbox_x0=bbox[1],
                bbox_y1=bbox[2],
                bbox_x1=bbox[3],
                box_top=cbox[0],
                box_left=cbox[1],
                box_bottom=cbox[2],
                box_right=cbox[3],
                status="ok"
            )

        except Exception as e:
            inf = {
                "key": r.key,
                "status": f"PREPROCESS_FAILED: {type(e).__name__}: {e}"
            }

        rows.append(inf)

        if (i + 1) % 20 == 0 or i + 1 == len(use):
            print(
                f"  preprocessed {i + 1}/{len(use)} "
                f"({time.time() - t0:.0f}s)"
            )

    prep = pd.DataFrame(rows)
    prep.to_csv(CACHE_TAB, index=False)
    np.savez_compressed(CACHE_NPZ, **canv)


# Stop if any image failed preprocessing.
failed = prep[prep["status"] != "ok"]

if len(failed):
    print("PREPROCESSING FAILURES:")
    print(failed[["key", "status"]].to_string(index=False))
    raise RuntimeError(
        "Some images failed preprocessing. "
        "Inspect them, then fix or exclude them explicitly."
    )

# Save preprocessing settings and image-level QC measures.
use = use.merge(prep, on="key", how="left", validate="one_to_one")
save_json(PREP_CFG, RESULTS / "tables" / "preprocess_config.json")
use.drop(columns=["flags_str"]).to_csv(
    RESULTS / "tables" / "preprocess_table.csv",
    index=False
)

print(
    "preprocessed", len(use), "images; canvas", (CANVAS, CANVAS),
    "| config hash", prep_hash
)


# ---- Similarity check on processed images ----
from skimage.transform import resize as _rs


def small_vec(c):
    """Create a standardised vector for image correlation."""
    v = _rs(
        c.astype(np.float32) / 255.0,
        (56, 56),
        anti_aliasing=True
    ).ravel()
    return (v - v.mean()) / (v.std() + 1e-6)


_keys = list(use["key"])
V = np.stack([small_vec(canv[k]) for k in _keys])
Rm = V @ V.T / V.shape[1]

lookup = use.set_index("key").reindex(_keys)
_pid = lookup["pair_id"].to_numpy()
_spl = lookup["split"].to_numpy()

# Compare each distinct image combination once, excluding the same shoe pair.
a_idx, b_idx = np.triu_indices(len(_keys), k=1)
different_pair = _pid[a_idx] != _pid[b_idx]
a_idx, b_idx = a_idx[different_pair], b_idx[different_pair]

similarities = pd.DataFrame({
    "r": Rm[a_idx, b_idx],
    "key_a": [_keys[a] for a in a_idx],
    "key_b": [_keys[b] for b in b_idx],
    "split_a": _spl[a_idx],
    "split_b": _spl[b_idx]
}).sort_values("r", ascending=False)

# Save the 15 most similar combinations for inspection.
nd = similarities.head(15)
nd.to_csv(
    RESULTS / "tables" / "near_duplicates_canvas_top15.csv",
    index=False
)

print(
    "\nMost similar images from different shoe pairs "
    f"(review r >= {NEAR_DUP_R:.2f}):"
)
print(nd.head(5).round(3).to_string(index=False))

# Check all combinations for high similarity across splits.
sus = similarities[
    (similarities["r"] >= NEAR_DUP_R)
    & (similarities["split_a"] != similarities["split_b"])
]
sus.to_csv(
    RESULTS / "tables" / "near_duplicates_canvas_cross_split.csv",
    index=False
)

if len(sus):
    print(
        f"WARNING: {len(sus)} highly similar image combinations cross splits. "
        "Review tables/near_duplicates_canvas_cross_split.csv."
    )
else:
    print(
        f"No cross-split image combinations have r >= {NEAR_DUP_R:.2f}. "
        "This does not rule out other forms of leakage."
    )

### 4.1 Visual quality control

Inspect a sample of training images from each class. The left panel shows the scan with the detected bounding box; the right shows the processed 224 × 224 image.

Check for cropped tread, scanner borders, rulers, text, stray marks, and unusual backgrounds. Any preprocessing adjustments are based on training images only.

Automated checks flag potential issues across all included images. These flags are recorded for review; they do not automatically exclude images or confirm their orientation.

In [ ]:
# Sample up to six training images from each class.
train_use = use[use["split"] == "train"].reset_index(drop=True)

sel = pd.concat([
    train_use[train_use["brand"] == c].sample(
        n=min(6, (train_use["brand"] == c).sum()),
        random_state=SEED
    )
    for c in CLASSES
])

# Display two scan/canvas comparisons per row.
n_rows = max(1, math.ceil(len(sel) / 2))
fig, axes = plt.subplots(
    n_rows, 4,
    figsize=(11, 3.1 * n_rows),
    squeeze=False
)

for n, (_, r) in enumerate(sel.iterrows()):
    a1 = axes[n // 2, (n % 2) * 2]
    a2 = axes[n // 2, (n % 2) * 2 + 1]

    # Use the same grayscale reader as preprocessing.
    g = Image.fromarray(
        read_gray_uint8(IMAGE_DIR / r.matched_path)
    ).reduce(8)
    a1.imshow(g, cmap="gray", vmin=0, vmax=255)

    # Scale the full-resolution bounding box to the preview.
    sc = 1 / 8
    a1.add_patch(plt.Rectangle(
        (r.bbox_x0 * sc, r.bbox_y0 * sc),
        (r.bbox_x1 - r.bbox_x0) * sc,
        (r.bbox_y1 - r.bbox_y0) * sc,
        fill=False,
        ec="r",
        lw=1.2
    ))

    a2.imshow(canv[r.key], cmap="gray", vmin=0, vmax=255)
    a1.set_title(f"{r.brand} pair {r.pair_id}{r.side}", fontsize=8)
    a2.set_title("processed image", fontsize=8)
    a1.axis("off")
    a2.axis("off")

# Hide unused panels.
for ax in axes.ravel()[len(sel) * 2:]:
    ax.axis("off")

plt.tight_layout()
plt.savefig(RESULTS / "figures" / "fig01_preprocessing_qc_train.png")
plt.show()


# Record descriptive QC flags without changing preprocessing settings.
qc = use.assign(
    flag_border_ink=use["border_ink_frac"] > 0.01,
    flag_many_components=use["n_components_kept"] > 3,
    flag_orientation_unclear=use["width_ratio_top_bottom"] < 1.05,
    flag_box_touches_scan_edge=(
        (use["bbox_y0"] <= 1)
        | (use["bbox_x0"] <= 1)
        | (use["bbox_y1"] >= use["raw_h"] - 1)
        | (use["bbox_x1"] >= use["raw_w"] - 1)
    )
)

# The width-ratio flag is a heuristic, not an orientation decision.
flag_cols = [c for c in qc.columns if c.startswith("flag_")]

print("Automated QC flag counts (all included images):")
print(qc[flag_cols].sum().to_string())

qc[
    ["key", "pair_id", "side", "brand", "split"] + flag_cols
].to_csv(
    RESULTS / "tables" / "preprocess_qc_flags.csv",
    index=False
)

### 4.2 Check image orientation

Inspect the training images for consistent orientation. The top-to-bottom width ratio and horizontal centre shift provide descriptive cues, but sparse tread patterns can make them unreliable. No images are rotated or flipped.

A contact sheet shows all training images for visual review. Occlusion regions retain the names `top_third`, `middle_third`, and `bottom_third`, because this check does not establish anatomical boundaries or confirm orientation across all splits.

In [ ]:
# Summarise orientation cues using training images only.
tr = use[use["split"] == "train"].copy()

tr["toe_up_heuristic"] = np.where(
    tr["width_ratio_top_bottom"] > 1.05,
    "top wider",
    np.where(
        tr["width_ratio_top_bottom"] < 0.95,
        "bottom wider",
        "unclear"
    )
)

# Describe the centre shift without assuming which side contains the arch.
tr["arch_side_heuristic"] = np.where(
    tr["waist_shift"] < -0.02,
    "centre shifts left",
    np.where(
        tr["waist_shift"] > 0.02,
        "centre shifts right",
        "unclear"
    )
)

print("TRAIN images — top/bottom width:")
print(pd.crosstab(tr["brand"], tr["toe_up_heuristic"]).to_string())

print("\nTRAIN images — centre shift by L/R label:")
print(pd.crosstab(tr["side"], tr["arch_side_heuristic"]).to_string())


# List training images requiring closer orientation review.
odd = tr.loc[
    tr["width_ratio_top_bottom"] < 1.05,
    ["key", "brand", "width_ratio_top_bottom"]
]
print("\nTraining images with unclear or bottom-wider cues:")
print(odd.to_string(index=False) if len(odd) else "none")

# Save descriptive measures for all included images.
use[[
    "key", "pair_id", "side", "brand", "split",
    "width_ratio_top_bottom", "waist_shift"
]].to_csv(
    RESULTS / "tables" / "orientation_heuristics.csv",
    index=False
)


# Display all training images for visual review.
ts = tr.sort_values(["brand", "pair_id", "side"])
ncol = 12
nrow = max(1, math.ceil(len(ts) / ncol))

fig, axs = plt.subplots(
    nrow, ncol,
    figsize=(ncol * 0.95, nrow * 1.25),
    squeeze=False
)
axs = axs.ravel()

for ax in axs:
    ax.axis("off")

for ax, (_, r) in zip(axs, ts.iterrows()):
    ax.imshow(canv[r.key], cmap="gray", vmin=0, vmax=255)
    ax.set_title(
        f"{r.brand[0]}{r.pair_id}{r.side}",
        fontsize=5,
        pad=1
    )

plt.tight_layout(pad=0.2)
plt.savefig(RESULTS / "figures" / "fig01b_train_contact_sheet.png")
plt.show()


# Use positional names rather than anatomical labels.
ORIENTATION_VERIFIED = False
REGION_NAMES = {
    "top": "top_third",
    "middle": "middle_third",
    "bottom": "bottom_third"
}
print("\nregion names in use:", REGION_NAMES)

## 5. Define the occlusion plan

Define the occlusion conditions before test evaluation. Mask percentages refer to the resized crop area, including its crop margin but excluding the outer canvas padding. Actual mask area and ink coverage are recorded.

| Condition | Definition |
|---|---|
| `unmodified` | No masking. |
| `top_third`, `middle_third`, `bottom_third` | Fill one third of the crop height across its full width. |
| `random_10`, `random_25`, `random_40` | Place a rectangle covering approximately 10%, 25%, or 40% of the crop area. |

Random rectangles use a log-uniform width-to-height ratio between 0.5 and 2.0, subject to fitting inside the crop. Positions are sampled uniformly. Pixel rounding and any clipping may change the realised mask area.

Each random condition has five draws per image. Fixed seeds ensure that every model receives the same masks. Metrics are calculated for each draw and then averaged; draws are not treated as independent test samples.

Masks are filled with white to simulate missing ink. Ink coverage is the fraction of total crop darkness covered by the mask. This controlled simulation does not reproduce all forms of crime-scene incompleteness.

In [ ]:
# Define unmodified, regional, and random occlusion conditions.
REGIONS = ["top", "middle", "bottom"]
CONDITIONS = (
    [("unmodified", "none", None)]
    + [(REGION_NAMES[r], "region", r) for r in REGIONS]
    + [
        (f"random_{int(p * 100)}", "random", p)
        for p in RANDOM_LEVELS
    ]
)
COND_NAMES = [c[0] for c in CONDITIONS]


def n_draws(kind):
    return K_MASKS if kind == "random" else 1


# Divide the crop into three horizontal regions.
def region_rect(box, which):
    top, left, bottom, right = box
    ch = bottom - top
    a = top + ch // 3
    b = top + 2 * ch // 3

    return {
        "top": (top, a),
        "middle": (a, b),
        "bottom": (b, bottom)
    }[which] + (left, right)


def random_rect(box, frac, rng):
    """Sample a rectangle with approximately the requested crop area."""
    top, left, bottom, right = box
    ch, cw = bottom - top, right - left
    area = frac * ch * cw

    # Sample aspect ratios until the rectangle fits.
    for _ in range(500):
        aspect = float(np.exp(rng.uniform(
            np.log(MASK_ASPECT_RANGE[0]),
            np.log(MASK_ASPECT_RANGE[1])
        )))
        h = math.sqrt(area / aspect)
        w = aspect * h

        if h <= ch and w <= cw:
            break
    else:
        # Clip if no feasible rectangle was found.
        h, w = min(h, ch), min(w, cw)

    # Round to whole pixels and sample a valid position.
    hi = max(1, min(int(round(h)), ch))
    wi = max(1, min(int(round(w)), cw))
    y = top + int(rng.integers(0, ch - hi + 1))
    x = left + int(rng.integers(0, cw - wi + 1))

    return y, y + hi, x, x + wi


def apply_condition(canvas_u8, box, cond, key, k=0):
    """Return the masked image and its realised coverage."""
    name, kind, par = cond
    x = canvas_u8.astype(np.float32) / 255.0

    info = {
        "condition": name,
        "draw": k,
        "key": key,
        "area_frac_of_box": 0.0,
        "ink_coverage": 0.0
    }

    if kind == "none":
        return x, info

    if kind == "region":
        y0, y1, x0, x1 = region_rect(box, par)
    else:
        # Reproduce masks using the image, level, and draw index.
        rng = np.random.default_rng([
            SEED,
            zlib.crc32(key.encode()),
            int(round(par * 100)),
            k
        ])
        y0, y1, x0, x1 = random_rect(box, par, rng)

    # Measure coverage before replacing pixels with white.
    ink = 1.0 - x
    total_ink = ink[box[0]:box[2], box[1]:box[3]].sum()
    box_area = (box[2] - box[0]) * (box[3] - box[1])

    info.update(
        rect=(y0, y1, x0, x1),
        area_frac_of_box=(
            (y1 - y0) * (x1 - x0) / box_area
        ),
        ink_coverage=float(
            ink[y0:y1, x0:x1].sum() / max(total_ink, 1e-6)
        )
    )

    x[y0:y1, x0:x1] = 1.0
    return x, info


# Locate the resized crop within the padded canvas.
def box_of(r):
    return (
        int(r.box_top), int(r.box_left),
        int(r.box_bottom), int(r.box_right)
    )


# Illustrate the conditions using one training image per class.
ex = [
    train_use[train_use["brand"] == c].iloc[0]
    for c in CLASSES
]
fig, axes = plt.subplots(
    len(ex), len(CONDITIONS),
    figsize=(1.9 * len(CONDITIONS), 2.1 * len(ex)),
    squeeze=False
)

for i, r in enumerate(ex):
    for j, cond in enumerate(CONDITIONS):
        xm, inf = apply_condition(
            canv[r.key], box_of(r), cond, r.key, k=0
        )
        axes[i, j].imshow(xm, cmap="gray", vmin=0, vmax=1)
        axes[i, j].axis("off")
        axes[i, j].set_title(
            f"{cond[0]}\n"
            f"{inf['area_frac_of_box']:.0%} area / "
            f"{inf['ink_coverage']:.0%} ink",
            fontsize=7
        )

plt.tight_layout()
plt.savefig(
    RESULTS / "figures" / "fig02_occlusion_conditions_train_example.png"
)
plt.show()


# Save the occlusion settings for the experimental plan.
plan = {
    "conditions": [
        {"name": c[0], "kind": c[1], "parameter": c[2]}
        for c in CONDITIONS
    ],
    "K_masks_random": K_MASKS,
    "mask_aspect_range": MASK_ASPECT_RANGE,
    "percent_refers_to": (
        "resized crop area, including crop margin "
        "but excluding outer canvas padding"
    ),
    "ink_coverage_definition": "fraction of total crop darkness masked",
    "fill": "white (1.0)",
    "seed": SEED,
    "orientation_verified": ORIENTATION_VERIFIED
}

save_json(plan, RESULTS / "tables" / "occlusion_plan.json")
print(json.dumps(plan, indent=1))

## 6. Train and select the models

Compare three models using the same preprocessed images. Training uses the training set, and model selection uses the validation set.

1. **Majority-class baseline:** always predict the most frequent training class. Ties select Adidas. Balanced accuracy is 0.5 when both classes are present in the evaluation set.
2. **HOG + logistic regression:** extract HOG features from 128 × 128 images and standardise them using training-set statistics. Select C by validation balanced accuracy, breaking ties with lower log-loss.
3. **Pretrained ResNet18:** freeze the ImageNet backbone and train a binary classification head. Use class-weighted loss and modest augmentation: rotation ±5°, translation ±5%, scale ±5%, contrast ±10%, and brightness offset ±0.05. No flips are applied.

Select CNN checkpoints by validation log-loss and the learning rate by mean validation log-loss across three seeds. Final CNN scores average the three selected models. Optional layer4 fine-tuning is disabled in the main experiment.

CNN scores have not been calibrated and should not be interpreted as validated probabilities. Training and validation losses also differ: training uses augmented images and weighted loss, while validation uses unmodified images and unweighted log-loss.

In [ ]:
# Retrieve images and labels for one split.
def get_split(name):
    d = use[use["split"] == name].reset_index(drop=True)
    X = np.stack([canv[k] for k in d["key"]])
    return d, X, d["y"].to_numpy()


tr_df, Xtr_u8, ytr = get_split("train")
va_df, Xva_u8, yva = get_split("val")

print(
    "train:", Xtr_u8.shape, np.bincount(ytr),
    "| val:", Xva_u8.shape, np.bincount(yva)
)

# Weight the positive class using the training class ratio.
class_ratio = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
print(
    f"training class counts: {np.bincount(ytr).tolist()} "
    f"-> pos_weight {class_ratio:.2f}"
)


# Convert scores to labels and calculate classification metrics.
def metrics_basic(y, p):
    yh = (p > 0.5).astype(int)
    return {
        "acc": float((yh == y).mean()),
        "bal_acc": float(balanced_accuracy_score(y, yh)),
        "logloss": float(log_loss(
            y, np.clip(p, 1e-6, 1 - 1e-6), labels=[0, 1]
        ))
    }


# ---- 6.1 Majority-class baseline ----
# argmax selects class 0 when class counts are tied.
maj_class = int(np.bincount(ytr, minlength=2).argmax())


class MajorityModel:
    name = "majority"

    def predict(self, canv_float):
        return np.full(len(canv_float), float(maj_class))


print(
    "majority class:", CLASSES[maj_class],
    "| validation:",
    metrics_basic(yva, MajorityModel().predict(Xva_u8))
)


# ---- 6.2 HOG + logistic regression ----
def hog_features(canv_float):
    """Extract local edge-orientation features from each image."""
    out = []
    for c in canv_float:
        im = np.asarray(
            Image.fromarray(c.astype(np.float32)).resize(
                (HOG_SIZE, HOG_SIZE), Image.BILINEAR
            )
        )
        out.append(feature.hog(
            im,
            orientations=9,
            pixels_per_cell=(16, 16),
            cells_per_block=(2, 2),
            block_norm="L2-Hys"
        ))
    return np.asarray(out)


Ftr = hog_features(Xtr_u8 / 255.0)
Fva = hog_features(Xva_u8 / 255.0)

hog_rows, best = [], None

for C in HOG_C_GRID:
    # Fit both the scaler and classifier on training data only.
    pipe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=C,
            max_iter=5000,
            class_weight="balanced"
        )
    ).fit(Ftr, ytr)

    m = metrics_basic(yva, pipe.predict_proba(Fva)[:, 1])
    hog_rows.append({"C": C, **m})

    # Prefer higher balanced accuracy, then lower log-loss.
    if best is None or (
        m["bal_acc"], -m["logloss"]
    ) > (
        best[1]["bal_acc"], -best[1]["logloss"]
    ):
        best = (C, m, pipe)

HOG_C, _, hog_pipe = best
hog_grid = pd.DataFrame(hog_rows)

print(hog_grid.to_string(index=False))
print("selected C =", HOG_C, "(by validation)")
hog_grid.to_csv(
    RESULTS / "tables" / "hog_validation_grid.csv",
    index=False
)


class HOGModel:
    name = "hog_logreg"

    def predict(self, canv_float):
        return hog_pipe.predict_proba(hog_features(canv_float))[:, 1]


# ---- 6.3 ResNet18 ----
from torchvision.transforms import v2 as T2

# ImageNet channel-normalisation values.
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class ResNetHead(nn.Module):
    def __init__(self, pretrained=True, unfreeze_layer4=False):
        super().__init__()
        weights = (
            torchvision.models.ResNet18_Weights.IMAGENET1K_V1
            if pretrained else None
        )
        self.backbone = torchvision.models.resnet18(weights=weights)

        # Replace the original classifier with a binary head.
        self.backbone.fc = nn.Identity()
        for p in self.backbone.parameters():
            p.requires_grad = False

        if unfreeze_layer4:
            for p in self.backbone.layer4.parameters():
                p.requires_grad = True

        self.head = nn.Linear(512, 1)

    def train(self, mode=True):
        super().train(mode)
        # Keep backbone BatchNorm statistics fixed.
        self.backbone.eval()
        return self

    def forward(self, x):
        assert x.is_cuda and all(
            p.is_cuda for p in self.parameters()
        ), "CNN forward pass must use CUDA."
        # Repeat grayscale across three channels and normalise.
        x = (
            x.repeat(1, 3, 1, 1) - MEAN.to(x.device)
        ) / STD.to(x.device)
        return self.head(self.backbone(x)).squeeze(1)


# Apply modest geometric changes with white fill and no flips.
AUG = T2.RandomAffine(
    degrees=5,
    translate=(0.05, 0.05),
    scale=(0.95, 1.05),
    fill=1.0
)


def augment(xb):
    """Augment training images independently."""
    out = torch.stack([AUG(x) for x in xb])
    contrast = 1 + (torch.rand(len(xb), 1, 1, 1) - 0.5) * 0.2
    brightness = (torch.rand(len(xb), 1, 1, 1) - 0.5) * 0.1
    return (
        (out - 0.5) * contrast + 0.5 + brightness
    ).clamp(0, 1)


# Convert arrays to tensors shaped (batch, channel, height, width).
def to_tensor(canv_float):
    return torch.from_numpy(
        np.asarray(canv_float, dtype=np.float32)
    ).unsqueeze(1)


# Predict in batches without computing gradients.
@torch.no_grad()
def predict_scores(model, canv_float, bs=32):
    model.eval()
    out = []
    for i in range(0, len(canv_float), bs):
        xb = to_tensor(canv_float[i:i + bs]).to(DEVICE)
        out.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(out)


def train_head(seed, lr, unfreeze=False):
    """Train one candidate and restore its best validation checkpoint."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = ResNetHead(PRETRAINED, unfreeze).to(DEVICE)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(
        params, lr=lr, weight_decay=WEIGHT_DECAY
    )

    # Use a smaller learning rate for optional layer4 training.
    if unfreeze:
        opt = torch.optim.Adam(
            [
                {"params": model.head.parameters(), "lr": lr},
                {
                    "params": model.backbone.layer4.parameters(),
                    "lr": 1e-4
                }
            ],
            weight_decay=WEIGHT_DECAY
        )

    lossf = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(float(class_ratio)).to(DEVICE)
    )
    Xt = to_tensor(Xtr_u8 / 255.0)
    yt = torch.from_numpy(ytr.astype(np.float32))
    Xv = Xva_u8 / 255.0

    hist, best_state = [], None
    best_loss, best_ep, bad = np.inf, -1, 0

    for ep in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(len(Xt))
        tl, tc = 0.0, 0

        # Shuffle training images and update trainable parameters.
        for i in range(0, len(Xt), BATCH):
            idx = perm[i:i + BATCH]
            xb = augment(Xt[idx]).to(DEVICE)
            yb = yt[idx].to(DEVICE)

            opt.zero_grad()
            logits = model(xb)
            loss = lossf(logits, yb)
            loss.backward()
            opt.step()

            tl += loss.item() * len(idx)
            tc += ((logits > 0).float() == yb).sum().item()

        # Evaluate validation images without augmentation.
        pv = predict_scores(model, Xv)
        mv = metrics_basic(yva, pv)
        hist.append({
            "epoch": ep + 1,
            "train_loss": tl / len(Xt),
            "train_acc": tc / len(Xt),
            "val_loss": mv["logloss"],
            "val_acc": mv["acc"],
            "val_bal_acc": mv["bal_acc"]
        })

        # Save the best checkpoint and stop after repeated non-improvement.
        if mv["logloss"] < best_loss - 1e-4:
            best_loss, best_ep, bad = mv["logloss"], ep + 1, 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
                if k.startswith("head") or (unfreeze and "layer4" in k)
            }
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best_state, strict=False)
    return model, pd.DataFrame(hist), {
        "best_val_loss": best_loss,
        "best_epoch": best_ep
    }


# ---- 6.4 Learning-rate selection ----
runs, hists = [], {}
t0 = time.time()

for lr in LR_GRID:
    for sd in SEEDS:
        mdl, h, bi = train_head(sd, lr)
        pv = predict_scores(mdl, Xva_u8 / 255.0)
        mv = metrics_basic(yva, pv)

        runs.append({
            "lr": lr,
            "seed": sd,
            **bi,
            "val_acc": mv["acc"],
            "val_bal_acc": mv["bal_acc"]
        })
        hists[(lr, sd)] = (mdl, h)

        print(
            f"lr={lr:g} seed={sd}: "
            f"best val loss {bi['best_val_loss']:.3f} "
            f"@ epoch {bi['best_epoch']} | "
            f"val bal-acc {mv['bal_acc']:.3f} "
            f"({time.time() - t0:.0f}s)"
        )

runs_df = pd.DataFrame(runs)
runs_df.to_csv(
    RESULTS / "tables" / "cnn_validation_runs.csv",
    index=False
)

# Select the learning rate by mean validation loss across seeds.
BEST_LR = float(
    runs_df.groupby("lr")["best_val_loss"].mean().idxmin()
)
print("selected learning rate:", BEST_LR)

cnn_models = [hists[(BEST_LR, sd)][0] for sd in SEEDS]

# Save the selected frozen models' heads.
for sd in SEEDS:
    torch.save(
        {
            k: v
            for k, v in hists[(BEST_LR, sd)][0].state_dict().items()
            if k.startswith("head")
        },
        RESULTS / "models" / f"resnet18_head_lr{BEST_LR:g}_seed{sd}.pt"
    )


class CNNEnsemble:
    name = "resnet18_frozen"

    def predict(self, canv_float):
        # Average scores rather than class labels.
        return np.mean([
            predict_scores(m, canv_float) for m in cnn_models
        ], axis=0)


# ---- Optional layer4 fine-tuning ----
# Train new candidates; adopt them only if validation meets the preset rule.
FINAL_VARIANT = "frozen_backbone"

if TRY_FINETUNE:
    ft = []
    for sd in SEEDS:
        m_ft, h_ft, bi_ft = train_head(sd, BEST_LR, unfreeze=True)
        pv = predict_scores(m_ft, Xva_u8 / 255.0)
        ft.append((m_ft, h_ft, bi_ft, metrics_basic(yva, pv)))

    base = runs_df[runs_df["lr"] == BEST_LR]
    d_loss = (
        base["best_val_loss"].mean()
        - np.mean([f[2]["best_val_loss"] for f in ft])
    )
    d_bal = (
        np.mean([f[3]["bal_acc"] for f in ft])
        - base["val_bal_acc"].mean()
    )

    print(
        f"layer4 fine-tuning: validation loss improvement {d_loss:+.3f}, "
        f"balanced-accuracy change {d_bal:+.3f}"
    )

    if d_loss >= FT_MIN_VAL_LOSS_IMPROVEMENT and d_bal >= 0:
        cnn_models = [f[0] for f in ft]
        FINAL_VARIANT = "finetune_layer4"

        for sd, f in zip(SEEDS, ft):
            hists[(BEST_LR, sd)] = (f[0], f[1])

        print("fine-tuned models selected.")
    else:
        print("frozen models retained.")

print("final CNN variant:", FINAL_VARIANT)


# ---- Learning curves ----
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))

for sd in SEEDS:
    h = hists[(BEST_LR, sd)][1]
    ax[0].plot(h["epoch"], h["train_loss"], "--", label=f"train s{sd}")
    ax[0].plot(h["epoch"], h["val_loss"], "-", label=f"val s{sd}")
    ax[1].plot(h["epoch"], h["train_acc"], "--", label=f"train s{sd}")
    ax[1].plot(h["epoch"], h["val_acc"], "-", label=f"val s{sd}")

ax[0].set(
    title="Loss (train: weighted BCE; val: unweighted log-loss)",
    xlabel="epoch"
)
ax[1].set(title="Accuracy (image level)", xlabel="epoch")
ax[0].legend(fontsize=7)

plt.tight_layout()
plt.savefig(RESULTS / "figures" / "fig03_learning_curves.png")
plt.show()

pd.concat([
    hists[(BEST_LR, sd)][1].assign(seed=sd)
    for sd in SEEDS
]).to_csv(
    RESULTS / "tables" / "cnn_learning_curves.csv",
    index=False
)

## 7. Record the experimental plan (Gate 3)

Before test evaluation, save the split hash, preprocessing settings, occlusion conditions, selected model settings, and model file hashes.

This record documents the final plan; it does not prevent code changes. After reviewing test results, any changes to the experiment must be reported as exploratory. Rewriting the lock does not create a new held-out test set.

In [ ]:
LOCK_PATH = RESULTS / "plan_lock.json"

# Avoid overwriting an existing record of test evaluation.
if LOCK_PATH.exists():
    previous_lock = json.loads(LOCK_PATH.read_text())
    if previous_lock.get("test_evaluated_at") is not None:
        raise RuntimeError(
            "Test evaluation is already recorded. Preserve this plan lock "
            "and document further experiments separately."
        )


# Save the actual selected CNN parameters.
weights = []

for sd, model in zip(SEEDS, cnn_models):
    weight_path = (
        RESULTS / "models"
        / f"resnet18_selected_{FINAL_VARIANT}_seed{sd}.pt"
    )

    # Frozen models need only the head; fine-tuned models also need layer4.
    state = {
        k: v.detach().cpu()
        for k, v in model.state_dict().items()
        if k.startswith("head")
        or (FINAL_VARIANT == "finetune_layer4" and "layer4" in k)
    }
    torch.save(state, weight_path)
    weights.append(weight_path)


# Save the fitted HOG pipeline, including its scaler and classifier.
import joblib

hog_path = RESULTS / "models" / "hog_logreg_selected.joblib"
joblib.dump(hog_pipe, hog_path)
weights.append(hog_path)


# Confirm that the split manifest has not changed.
assert sha256_file(MAN_PATH) == MAN_SHA, (
    "split manifest changed; review before test evaluation"
)


# Record the selected settings and model hashes.
lock = {
    "locked_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "split_manifest_sha256": MAN_SHA,
    "preprocessing": PREP_CFG,
    "occlusion_plan": plan,
    "majority_class": maj_class,
    "hog_C": HOG_C,
    "HOG_SIZE": HOG_SIZE,
    "cnn_lr": BEST_LR,
    "cnn_variant": FINAL_VARIANT,
    "seeds": list(SEEDS),
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "batch_size": BATCH,
    "weight_decay": WEIGHT_DECAY,
    "pretrained": PRETRAINED,
    "model_weight_sha256": {
        w.name: sha256_file(w) for w in weights
    },
    "bootstrap_B": B_BOOT,
    "ci_percentiles": CI,
    "n_test_pairs": int(
        use.loc[use["split"] == "test", "pair_id"].nunique()
    ),
    "test_evaluated_at": None
}

save_json(lock, LOCK_PATH)

print("plan lock written:", LOCK_PATH.relative_to(ROOT))
print(json.dumps({
    k: lock[k]
    for k in ["hog_C", "cnn_lr", "cnn_variant", "n_test_pairs"]
}, indent=2))

## 8. Evaluate the held-out test set

Check the saved plan and model files before evaluation. Every model receives the same masked test images. Predictions, mask coverage, and individual CNN seed scores are saved for analysis.

This cell records when test evaluation starts and finishes. Repeated evaluation does not provide new independent evidence.

In [ ]:
# Load the plan and check that test evaluation has not already started.
if not LOCK_PATH.exists():
    raise RuntimeError("Plan lock missing; run Section 7.")

lock = json.loads(LOCK_PATH.read_text())

if (
    lock.get("test_evaluated_at") is not None
    or lock.get("test_evaluation_started_at") is not None
):
    raise RuntimeError(
        "Test evaluation has already started or finished. "
        "Preserve the record and use saved results where available."
    )

# Verify the current split manifest and saved model files.
assert sha256_file(MAN_PATH) == lock["split_manifest_sha256"], (
    "split manifest changed"
)

for name, expected_hash in lock["model_weight_sha256"].items():
    path = RESULTS / "models" / name
    assert path.exists() and sha256_file(path) == expected_hash, (
        f"model file missing or changed: {name}"
    )

# Check that the active experiment settings match the saved plan.
assert lock["preprocessing"] == json.loads(json.dumps(PREP_CFG)), (
    "preprocessing settings changed"
)
assert lock["occlusion_plan"] == json.loads(json.dumps(plan)), (
    "occlusion settings changed"
)
assert (
    lock["hog_C"] == HOG_C
    and lock["cnn_lr"] == BEST_LR
    and lock["cnn_variant"] == FINAL_VARIANT
    and lock["seeds"] == list(SEEDS)
), "selected model settings changed"


# Prepare test records and pair indices.
te_df, Xte_u8, yte = get_split("test")
te_df["pair_index"] = pd.factorize(te_df["pair_id"])[0]
n_te = len(te_df)
n_pairs_te = te_df["pair_id"].nunique()

print(
    f"TEST: {n_te} images from {n_pairs_te} shoe pairs "
    f"({te_df.drop_duplicates('pair_id')['brand'].value_counts().to_dict()} "
    "pairs per class)"
)

MODELS = [MajorityModel(), HOGModel(), CNNEnsemble()]

# Store Nike scores as arrays shaped (draws, test images).
PRED = {m.name: {} for m in MODELS}
mask_rows, per_seed_unmod = [], {}
t0 = time.time()

# Record the start so an interrupted run is not silently repeated.
lock["test_evaluation_started_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
save_json(lock, LOCK_PATH)


for cond in CONDITIONS:
    K = n_draws(cond[1])
    batches = []

    # Generate each masked batch once for all models.
    for k in range(K):
        xs = []
        for _, r in te_df.iterrows():
            xm, inf = apply_condition(
                canv[r.key], box_of(r), cond, r.key, k
            )
            xs.append(xm)
            mask_rows.append({**inf, "split": "test"})

        batches.append(np.stack(xs))

    for m in MODELS:
        PRED[m.name][cond[0]] = np.stack([
            m.predict(batch) for batch in batches
        ])

    # Save individual seed scores on unmodified images.
    if cond[0] == "unmodified":
        per_seed_unmod = {
            f"seed{sd}": predict_scores(cnn_models[j], batches[0])
            for j, sd in enumerate(SEEDS)
        }

    print(
        f"  {cond[0]:>15s}: {K} draw(s) evaluated "
        f"({time.time() - t0:.0f}s)"
    )


# Save one row per model, condition, draw, and image.
long = []

for m in MODELS:
    for c in COND_NAMES:
        for k in range(PRED[m.name][c].shape[0]):
            long.append(pd.DataFrame({
                "model": m.name,
                "condition": c,
                "draw": k,
                "key": te_df["key"].to_numpy(),
                "pair_id": te_df["pair_id"].to_numpy(),
                "side": te_df["side"].to_numpy(),
                "brand": te_df["brand"].to_numpy(),
                "y_true": yte,
                "score_nike": PRED[m.name][c][k]
            }))

long = pd.concat(long, ignore_index=True)
long["y_pred"] = (long["score_nike"] > 0.5).astype(int)
long.to_csv(
    RESULTS / "tables" / "test_predictions_long.csv",
    index=False
)


# Save mask coordinates and realised coverage.
mlog = pd.DataFrame(mask_rows)
mlog.assign(rect=mlog["rect"].astype(str)).to_csv(
    RESULTS / "tables" / "test_occlusion_mask_log.csv",
    index=False
)

coverage_cols = ["area_frac_of_box", "ink_coverage"]
print("\nMean realised mask area and ink coverage:")
print(
    mlog.groupby("condition")[coverage_cols]
    .mean().round(3).loc[COND_NAMES].to_string()
)

mlog.groupby("condition")[coverage_cols].describe().to_csv(
    RESULTS / "tables" / "mask_realised_coverage.csv"
)

pd.DataFrame(per_seed_unmod).assign(
    key=te_df["key"].to_numpy(),
    y_true=yte
).to_csv(
    RESULTS / "tables" / "cnn_per_seed_unmodified_scores.csv",
    index=False
)


# Mark completion after prediction files have been saved.
lock["test_evaluated_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
save_json(lock, LOCK_PATH)

## 9. Calculate metrics and uncertainty

Report accuracy, balanced accuracy, macro F1, and recall for each class at two levels:

- **Image level:** each left or right impression receives a prediction.
- **Shoe-pair level:** average the two images' scores before classification.

Calculate confidence intervals by resampling shoe pairs within each class, keeping both images together. Metrics are averaged across random-mask draws within each bootstrap replicate. Draws are not independent samples.

Compare occluded and unmodified results using the same bootstrap samples. A change interval containing zero is inconclusive; it does not establish that occlusion has no effect.

The test set is small. Bootstrap intervals can collapse to a single value when predictions are uniformly correct or incorrect, so they do not fully capture uncertainty at ceiling performance.

In [ ]:
def boot_metrics(y, yhat, W):
    """Calculate metrics for each bootstrap sample and mask draw."""
    # yhat: (draws, images or pairs); W: (resamples, images or pairs).
    y1 = (y == 1)[None, :]
    y0 = ~y1
    p1 = yhat == 1
    p0 = ~p1

    tp1 = W @ (y1 & p1).T
    fn1 = W @ (y1 & p0).T
    tn = W @ (y0 & p0).T
    fp = W @ (y0 & p1).T
    N = tp1 + fn1 + tn + fp

    # Assign zero when a metric denominator is zero.
    def divide(a, b):
        return np.divide(
            a, b,
            out=np.zeros_like(a, dtype=float),
            where=b > 0
        )

    rec1 = divide(tp1, tp1 + fn1)
    rec0 = divide(tn, tn + fp)
    prec1 = divide(tp1, tp1 + fp)
    prec0 = divide(tn, tn + fn1)
    f1_1 = divide(2 * prec1 * rec1, prec1 + rec1)
    f1_0 = divide(2 * prec0 * rec0, prec0 + rec0)

    return {
        "accuracy": divide(tp1 + tn, N),
        "balanced_accuracy": (rec0 + rec1) / 2,
        "macro_f1": (f1_0 + f1_1) / 2,
        f"recall_{CLASSES[0]}": rec0,
        f"recall_{CLASSES[1]}": rec1
    }


METRICS = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    f"recall_{CLASSES[0]}",
    f"recall_{CLASSES[1]}"
]


def make_weights(pair_labels, B, rng):
    """Count how often each pair is drawn in each bootstrap sample."""
    n = len(pair_labels)
    Wp = np.zeros((B, n))
    idx_by_class = [
        np.where(pair_labels == c)[0]
        for c in np.unique(pair_labels)
    ]

    # Resample within classes to preserve their pair counts.
    for b in range(B):
        for idx in idx_by_class:
            drawn = rng.choice(idx, size=len(idx), replace=True)
            Wp[b] += np.bincount(drawn, minlength=n)

    return Wp


pair_class = (
    te_df.drop_duplicates("pair_index")
    .sort_values("pair_index")["y"].to_numpy()
)
pair_idx_img = te_df["pair_index"].to_numpy()

# Give both images of a pair the same bootstrap multiplicity.
rng_boot = np.random.default_rng(SEED + 1)
Wp = make_weights(pair_class, B_BOOT, rng_boot)
Wi = Wp[:, pair_idx_img]

# Unit weights calculate metrics on the original test sample.
ones_i = np.ones((1, n_te))
ones_p = np.ones((1, n_pairs_te))


def pair_scores(sc):
    """Average left/right scores within each pair for each draw."""
    out = np.zeros((sc.shape[0], n_pairs_te))
    for p in range(n_pairs_te):
        out[:, p] = sc[:, pair_idx_img == p].mean(axis=1)
    return out


def summarize(level):
    """Calculate estimates and pair-bootstrap percentile intervals."""
    rows = []
    condition_kinds = {c[0]: c[1] for c in CONDITIONS}

    for m in MODELS:
        for c in COND_NAMES:
            sc = PRED[m.name][c]

            if level == "image":
                y = yte
                yh = (sc > 0.5).astype(int)
                W, W1 = Wi, ones_i
            else:
                y = pair_class
                yh = (pair_scores(sc) > 0.5).astype(int)
                W, W1 = Wp, ones_p

            point = boot_metrics(y, yh, W1)
            boot = boot_metrics(y, yh, W)

            for mt in METRICS:
                # Average draw-specific metrics within each resample.
                bm = boot[mt].mean(axis=1)
                lo, hi = np.percentile(bm, CI)

                rows.append({
                    "level": level,
                    "model": m.name,
                    "condition": c,
                    "metric": mt,
                    "estimate": float(point[mt].mean()),
                    "ci_low": lo,
                    "ci_high": hi,
                    "n_units": n_te if level == "image" else n_pairs_te,
                    "n_independent_pairs": n_pairs_te,
                    "draws": n_draws(condition_kinds[c])
                })

    return pd.DataFrame(rows)


res_img = summarize("image")
res_pair = summarize("pair")

res_img.to_csv(
    RESULTS / "tables" / "metrics_image_level.csv",
    index=False
)
res_pair.to_csv(
    RESULTS / "tables" / "metrics_pair_level.csv",
    index=False
)


# Display estimates and intervals in a compact table.
def show(res, title):
    t = res[res["metric"].isin([
        "accuracy", "balanced_accuracy", "macro_f1"
    ])].copy()
    t["value"] = t.apply(
        lambda r: (
            f"{r.estimate:.3f} "
            f"[{r.ci_low:.3f}, {r.ci_high:.3f}]"
        ),
        axis=1
    )

    table = t.pivot_table(
        index=["model", "condition"],
        columns="metric",
        values="value",
        aggfunc="first"
    )
    order = [(m.name, c) for m in MODELS for c in COND_NAMES]

    print(
        f"\n{title} "
        f"(95% pair-bootstrap interval; {n_pairs_te} test pairs)"
    )
    print(table.loc[order].to_string())


show(res_img, "IMAGE-LEVEL metrics")
show(res_pair, "SHOE-PAIR-LEVEL metrics")
print(
    "\nIntervals with identical endpoints may reflect ceiling performance, "
    "not absence of uncertainty."
)


# ---- Paired changes from unmodified performance ----
delta_rows = []

for m in MODELS:
    base_hat = (PRED[m.name]["unmodified"] > 0.5).astype(int)
    pb = boot_metrics(yte, base_hat, ones_i)
    bb = boot_metrics(yte, base_hat, Wi)
    corr_base = (base_hat[0] == yte).astype(float)

    for c in COND_NAMES[1:]:
        yh = (PRED[m.name][c] > 0.5).astype(int)
        pc = boot_metrics(yte, yh, ones_i)
        bc = boot_metrics(yte, yh, Wi)

        # Compare each image's correctness averaged across draws.
        corr = (yh == yte[None, :]).mean(axis=0)

        for mt in ["accuracy", "balanced_accuracy", "macro_f1"]:
            dpt = float(pc[mt].mean() - pb[mt].mean())

            # Use identical resamples for both conditions.
            db = bc[mt].mean(axis=1) - bb[mt].mean(axis=1)
            lo, hi = np.percentile(db, CI)

            if hi < 0:
                verdict = "interval excludes zero: decrease"
            elif lo > 0:
                verdict = "interval excludes zero: increase"
            else:
                verdict = "interval includes zero: inconclusive"

            delta_rows.append({
                "model": m.name,
                "condition": c,
                "metric": mt,
                "delta": dpt,
                "ci_low": lo,
                "ci_high": hi,
                "reading": verdict,
                "images_worse": int((corr < corr_base).sum()),
                "images_better": int((corr > corr_base).sum()),
                "images_same": int((corr == corr_base).sum())
            })

deltas = pd.DataFrame(delta_rows)
deltas.to_csv(
    RESULTS / "tables" / "paired_changes_vs_unmodified_image_level.csv",
    index=False
)

print("\nPAIRED changes in image-level balanced accuracy:")
print(
    deltas.loc[
        deltas["metric"] == "balanced_accuracy",
        [
            "model", "condition", "delta", "ci_low", "ci_high",
            "images_worse", "images_better", "reading"
        ]
    ].round(3).to_string(index=False)
)


# ---- Confusion counts averaged across draws ----
cm_rows = []

for m in MODELS:
    for c in COND_NAMES:
        yh = (PRED[m.name][c] > 0.5).astype(int)

        for t in (0, 1):
            for p in (0, 1):
                count = (
                    (yte[None, :] == t) & (yh == p)
                ).sum(axis=1).mean()

                cm_rows.append({
                    "model": m.name,
                    "condition": c,
                    "true": CLASSES[t],
                    "predicted": CLASSES[p],
                    "mean_count": float(count)
                })

cm = pd.DataFrame(cm_rows)
cm.to_csv(
    RESULTS / "tables" / "confusion_matrices.csv",
    index=False
)


# Report seed-specific accuracy on unmodified images.
print(
    "\nCNN accuracy by seed on unmodified test images:",
    {
        k: round(float(((v > 0.5).astype(int) == yte).mean()), 3)
        for k, v in per_seed_unmod.items()
    }
)

## 10. Visualise the results

Plot performance by random-mask size and mask location, paired changes from unmodified performance, CNN confusion matrices, and example predictions.

Error bars show 95% shoe-pair bootstrap intervals. Collapsed intervals at perfect performance do not imply certainty. Example images illustrate individual predictions and are not a representative sample of performance.

In [ ]:
# Use consistent colours and readable model names.
colors = {
    "majority": "tab:gray",
    "hog_logreg": "tab:orange",
    "resnet18_frozen": "tab:blue"
}
labels = {
    "majority": "Majority baseline",
    "hog_logreg": "HOG + logistic regression",
    "resnet18_frozen": (
        "ResNet18 (frozen)"
        if FINAL_VARIANT == "frozen_backbone"
        else "ResNet18 (layer4 fine-tuned)"
    )
}


def get(res, model, condition, metric):
    """Retrieve an estimate and its confidence limits."""
    r = res.loc[
        (res["model"] == model)
        & (res["condition"] == condition)
        & (res["metric"] == metric)
    ].iloc[0]
    return r.estimate, r.ci_low, r.ci_high


# Draw interval endpoints directly; percentile intervals need not contain
# the original estimate.
def draw_intervals(ax, x, low, high, color, cap=0.08):
    ax.vlines(x, low, high, color=color, linewidth=1)
    ax.hlines(low, x - cap, x + cap, color=color, linewidth=1)
    ax.hlines(high, x - cap, x + cap, color=color, linewidth=1)


# ---- Figure 4: Random-mask size ----
sev = [("unmodified", 0)] + [
    (f"random_{int(p * 100)}", int(p * 100))
    for p in RANDOM_LEVELS
]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)

for ax, res, title in zip(
    axes,
    [res_img, res_pair],
    [
        f"Image level ({n_te} images, {n_pairs_te} pairs)",
        f"Shoe-pair level ({n_pairs_te} pairs)"
    ]
):
    for j, m in enumerate(MODELS):
        e = np.array([
            get(res, m.name, c, "balanced_accuracy")
            for c, _ in sev
        ])
        x = np.array([s for _, s in sev]) + (j - 1) * 0.8

        ax.plot(
            x, e[:, 0], marker="o",
            label=labels[m.name], color=colors[m.name]
        )
        draw_intervals(
            ax, x, e[:, 1], e[:, 2], colors[m.name], cap=0.35
        )

    ax.axhline(0.5, color="k", lw=0.6, ls=":")
    ax.set(
        title=title,
        xlabel="Target random-mask area (% of crop)",
        ylim=(0, 1.05)
    )
    ax.set_xticks([s for _, s in sev])

axes[0].set_ylabel("Balanced accuracy (95% pair-bootstrap interval)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(
    RESULTS / "figures" / "fig04_balanced_accuracy_vs_random_mask_size.png"
)
plt.show()


# ---- Figure 5: Regional masks ----
reg = ["unmodified"] + [REGION_NAMES[r] for r in REGIONS]
fig, ax = plt.subplots(figsize=(7.5, 3.8))
w = 0.25

for j, m in enumerate(MODELS):
    e = np.array([
        get(res_img, m.name, c, "balanced_accuracy")
        for c in reg
    ])
    x = np.arange(len(reg)) + (j - 1) * w

    ax.bar(
        x, e[:, 0], width=w,
        label=labels[m.name], color=colors[m.name]
    )
    draw_intervals(ax, x, e[:, 1], e[:, 2], "black", cap=0.04)

ax.axhline(0.5, color="k", lw=0.6, ls=":")
ax.set_xticks(range(len(reg)))
ax.set_xticklabels(reg)
ax.set(
    ylabel="Balanced accuracy (image level)",
    ylim=(0, 1.05),
    title="Regional masking (approximately one third of the crop)"
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(
    RESULTS / "figures" / "fig05_balanced_accuracy_region_masking.png"
)
plt.show()


# ---- Figure 6: Paired changes ----
d = deltas[deltas["metric"] == "balanced_accuracy"]
conds = COND_NAMES[1:]
fig, ax = plt.subplots(figsize=(8, 3.8))

for j, m in enumerate(MODELS):
    dd = d[d["model"] == m.name].set_index("condition").loc[conds]
    x = np.arange(len(conds)) + (j - 1) * 0.15

    ax.plot(
        x, dd["delta"], "o",
        label=labels[m.name], color=colors[m.name]
    )
    draw_intervals(
        ax, x,
        dd["ci_low"].to_numpy(),
        dd["ci_high"].to_numpy(),
        colors[m.name],
        cap=0.04
    )

ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(range(len(conds)))
ax.set_xticklabels(conds, rotation=20, ha="right")
ax.set(
    ylabel="Change in balanced accuracy",
    title="Image-level change from unmodified performance"
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(
    RESULTS / "figures" / "fig06_paired_change_vs_unmodified.png"
)
plt.show()


# ---- Figure 7: CNN confusion matrices ----
# Use the same colour scale across all conditions.
cnn_cm = cm[cm["model"] == "resnet18_frozen"]
vmax = max(float(cnn_cm["mean_count"].max()), 1)

fig, axes = plt.subplots(
    1, len(COND_NAMES),
    figsize=(2.2 * len(COND_NAMES), 2.6),
    squeeze=False
)

for ax, c in zip(axes.ravel(), COND_NAMES):
    M = (
        cnn_cm[cnn_cm["condition"] == c]
        .pivot(index="true", columns="predicted", values="mean_count")
        .reindex(index=CLASSES, columns=CLASSES)
        .to_numpy()
    )
    ax.imshow(M, cmap="Blues", vmin=0, vmax=vmax)
    ax.set_title(c, fontsize=8)

    for (i, j), value in np.ndenumerate(M):
        text = f"{value:.1f}" if c.startswith("random") else f"{value:.0f}"
        ax.text(
            j, i, text,
            ha="center", va="center", fontsize=8,
            color="white" if value > vmax / 2 else "black"
        )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(CLASSES, fontsize=7)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(
        CLASSES if c == COND_NAMES[0] else ["", ""],
        fontsize=7
    )

fig.suptitle(
    f"{labels['resnet18_frozen']}: rows = true, columns = predicted\n"
    "Random conditions show mean counts across draws",
    fontsize=8
)
plt.tight_layout()
plt.savefig(RESULTS / "figures" / "fig07_confusion_matrices_cnn.png")
plt.show()


# ---- Figure 8: Example CNN predictions ----
# Select the first three correct and incorrect images for two conditions.
example_conditions = ["unmodified", "random_25"]
condition_lookup = {c[0]: c for c in CONDITIONS}
fig, axes = plt.subplots(2, 6, figsize=(12, 5.2))

for row, name in enumerate(example_conditions):
    scores = PRED["resnet18_frozen"][name][0]
    predicted = (scores > 0.5).astype(int)

    correct = np.where(predicted == yte)[0][:3]
    incorrect = np.where(predicted != yte)[0][:3]
    selected = (
        list(correct) + [None] * (3 - len(correct))
        + list(incorrect) + [None] * (3 - len(incorrect))
    )

    for col, (ax, i) in enumerate(zip(axes[row], selected)):
        ax.axis("off")

        if i is None:
            ax.text(
                0.5, 0.5,
                "No further correct examples" if col < 3
                else "No further incorrect examples",
                ha="center", va="center",
                transform=ax.transAxes, fontsize=7
            )
            continue

        r = te_df.iloc[i]
        xm, _ = apply_condition(
            canv[r.key], box_of(r), condition_lookup[name], r.key, k=0
        )
        is_correct = predicted[i] == yte[i]

        ax.imshow(xm, cmap="gray", vmin=0, vmax=1)
        ax.set_title(
            f"{'Correct' if is_correct else 'Incorrect'}: true {r.brand}\n"
            f"Nike score = {scores[i]:.2f}\n{name}",
            fontsize=7,
            color="g" if is_correct else "r"
        )

fig.suptitle(
    "Example test predictions: first three correct / incorrect\n"
    "Scores are uncalibrated; random_25 shows draw 0",
    fontsize=8
)
plt.tight_layout()
plt.savefig(RESULTS / "figures" / "fig08_example_predictions.png")
plt.show()

## 11. Explore potential shortcut cues

These analyses were developed after test review and are reported separately from the main experiment.

**S1: Descriptor probes.** Train logistic regression models using coarse geometry and scan/session descriptors. Select C on validation data and evaluate on the original test pairs with pair-bootstrap intervals.

- Geometry: original impression-box dimensions, aspect ratio, and ink coverage.
- Scan/session: paper brightness, background variation, scan dimensions, and collection date.

Above-chance performance suggests that class-related information is available in these descriptors. It does not establish that the CNN uses
them. Some descriptors, such as shape and ink coverage, also reflect the impression itself.

**S2: Shared-size subset.** Compare grouped cross-validation results using all pairs and subsets defined by shoe-size labels. Extract features with an ImageNet-pretrained, frozen ResNet18 and fit logistic regression with a fixed C of 0.1.

Pairs remain together in each fold. This analysis includes original test pairs in cross-validation training folds and is not a new held-out result. Shared shoe sizes reduce one source of confounding but do not balance size distributions or remove other shortcuts. Bootstrap intervals describe variation in the saved out-of-fold predictions, not the full uncertainty from retraining models.

In [ ]:
# ---- S1: Descriptor probes ----
def probe(cols, name):
    """Select a descriptor-only classifier using validation data."""
    Xa = tr_df[cols].to_numpy(dtype=float)
    Xv = va_df[cols].to_numpy(dtype=float)
    Xt = te_df[cols].to_numpy(dtype=float)

    # Stop if missing or invalid values require a preprocessing decision.
    if not all(np.isfinite(X).all() for X in (Xa, Xv, Xt)):
        raise ValueError(f"Missing or non-finite descriptors in {name}.")

    best = None

    for C in HOG_C_GRID:
        pipe = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=C,
                max_iter=5000,
                class_weight="balanced"
            )
        ).fit(Xa, ytr)

        m = metrics_basic(yva, pipe.predict_proba(Xv)[:, 1])

        if best is None or (
            m["bal_acc"], -m["logloss"]
        ) > (
            best[1]["bal_acc"], -best[1]["logloss"]
        ):
            best = (C, m, pipe)

    # Reuse the main experiment's shoe-pair bootstrap samples.
    yh = (
        best[2].predict_proba(Xt)[:, 1] > 0.5
    ).astype(int)[None, :]

    point = boot_metrics(yte, yh, ones_i)
    boot = boot_metrics(yte, yh, Wi)
    lo, hi = np.percentile(
        boot["balanced_accuracy"].mean(axis=1), CI
    )

    return {
        "probe": name,
        "C": best[0],
        "val_bal_acc": best[1]["bal_acc"],
        "test_bal_acc": float(point["balanced_accuracy"].mean()),
        "ci_low": lo,
        "ci_high": hi,
        "test_accuracy": float(point["accuracy"].mean())
    }


GEOM = [
    "bbox_h_px", "bbox_w_px",
    "bbox_aspect_w_over_h", "ink_frac_in_box"
]
SCAN = [
    "paper_level", "bg_std",
    "raw_h", "raw_w", "date_ordinal"
]

s1 = pd.DataFrame([
    probe(GEOM, "geometry"),
    probe(SCAN, "scan/session"),
    probe(GEOM + SCAN, "geometry + scan/session")
])
s1.to_csv(
    RESULTS / "tables" / "sensitivity_S1_shortcut_probes.csv",
    index=False
)

print("S1: exploratory descriptor probes (image-level metrics):")
print(s1.round(3).to_string(index=False))


# ---- S2: Grouped CV using frozen ImageNet features ----
@torch.no_grad()
def embed(canv_float, bs=32):
    """Extract pretrained features without fitting to this dataset."""
    backbone = torchvision.models.resnet18(
        weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1
    )
    backbone.fc = nn.Identity()
    backbone = backbone.to(DEVICE).eval()

    out = []
    for i in range(0, len(canv_float), bs):
        xb = to_tensor(canv_float[i:i + bs]).to(DEVICE)
        xb = (
            xb.repeat(1, 3, 1, 1) - MEAN.to(DEVICE)
        ) / STD.to(DEVICE)
        out.append(backbone(xb).cpu().numpy())

    return np.concatenate(out)


allk = use.reset_index(drop=True)
E = embed(
    np.stack([canv[k] for k in allk["key"]]) / 255.0
)


# Require consistent non-missing shoe-size labels within each pair.
size_counts = allk.groupby("pair_id")["shoe_size"].nunique()
assert (size_counts <= 1).all(), (
    "conflicting shoe-size labels within a pair"
)

pair_size = (
    allk.groupby("pair_id")
    .agg(
        brand=("brand", "first"),
        size=(
            "shoe_size",
            lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan
        )
    )
    .reset_index()
)

# Identify exact size labels represented in both classes.
shared = (
    set(pair_size.loc[
        pair_size["brand"] == CLASSES[0], "size"
    ].dropna())
    & set(pair_size.loc[
        pair_size["brand"] == CLASSES[1], "size"
    ].dropna())
)
shared_labels = sorted(shared, key=str)

print(
    "\nS2: pairs with unknown size:",
    int(pair_size["size"].isna().sum()),
    "| shared size labels:", shared_labels
)
print(pd.crosstab(
    pair_size["size"].fillna("unknown"),
    pair_size["brand"]
).to_string())


def cv_eval(pair_ids, label, n_splits=5):
    """Evaluate fixed-C logistic regression with shoe-pair grouped CV."""
    mask = allk["pair_id"].isin(pair_ids).to_numpy()
    d = allk.loc[mask].reset_index(drop=True)
    Ed = E[mask]
    y = d["y"].to_numpy()
    groups = d["pair_id"].to_numpy()

    per_class = d.drop_duplicates("pair_id")["brand"].value_counts()

    if len(per_class) < 2 or per_class.min() < 4:
        return {
            "subset": label,
            "n_pairs": int(d["pair_id"].nunique()),
            "note": "too few pairs per class for CV"
        }

    n_splits = int(min(n_splits, per_class.min()))
    oof = np.full(len(d), np.nan)

    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=SEED
    )

    for trn, tst in cv.split(Ed, y, groups):
        # Verify separation and fit scaling within each training fold.
        assert set(groups[trn]).isdisjoint(set(groups[tst]))
        assert len(np.unique(y[trn])) == 2, (
            "a CV training fold lacks one class"
        )

        pipe = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=0.1,
                max_iter=5000,
                class_weight="balanced"
            )
        ).fit(Ed[trn], y[trn])

        oof[tst] = pipe.predict_proba(Ed[tst])[:, 1]

    assert np.isfinite(oof).all(), "some images lack OOF predictions"

    # Bootstrap fixed OOF predictions by shoe pair.
    pid, _ = pd.factorize(groups)
    pc = (
        d.assign(pair_index=pid)
        .drop_duplicates("pair_index")
        .sort_values("pair_index")["y"].to_numpy()
    )
    W = make_weights(
        pc, B_BOOT, np.random.default_rng(SEED + 2)
    )[:, pid]

    yh = (oof > 0.5).astype(int)[None, :]
    point = boot_metrics(y, yh, np.ones((1, len(y))))
    boot = boot_metrics(y, yh, W)
    lo, hi = np.percentile(
        boot["balanced_accuracy"].mean(axis=1), CI
    )

    return {
        "subset": label,
        "n_pairs": int(d["pair_id"].nunique()),
        "pairs_per_class": per_class.to_dict(),
        "folds": n_splits,
        "bal_acc": float(point["balanced_accuracy"].mean()),
        "ci_low": lo,
        "ci_high": hi
    }


sized = pair_size.dropna(subset=["size"])
s2_rows = [cv_eval(pair_size["pair_id"], "all pairs")]

if shared:
    s2_rows.append(cv_eval(
        sized.loc[sized["size"].isin(shared), "pair_id"],
        f"shared-size labels only ({shared_labels})"
    ))
    s2_rows.append(cv_eval(
        sized.loc[~sized["size"].isin(shared), "pair_id"],
        "class-exclusive size labels"
    ))
else:
    print("No shared size labels; shared-size analysis skipped.")

s2 = pd.DataFrame(s2_rows)
s2.to_csv(
    RESULTS / "tables" / "sensitivity_S2_size_matched_cv.csv",
    index=False
)

print("\nS2: exploratory grouped CV (image-level metrics):")
print(s2.round(3).to_string(index=False))

## 12. Export the results

Save software versions and a machine-readable results summary. Package the audit tables, figures, selected model files, experimental plan, and split manifest into one ZIP file.

The archive excludes source TIFFs and cached image arrays. Figures include image previews and example predictions.

In [ ]:
# Record software versions and runtime settings.
env = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "scikit_image": __import__("skimage").__version__,
    "scipy": __import__("scipy").__version__,
    "pillow": __import__("PIL").__version__,
    "matplotlib": matplotlib.__version__,
    "joblib": joblib.__version__,
    "device": DEVICE,
    "gpu_name": torch.cuda.get_device_name(0),
    "purpose": "Exploratory GPU reproduction of previously examined data",
    "pretrained_weights": PRETRAINED,
    "smoke_mode": SMOKE
}
save_json(env, RESULTS / "environment.json")


# Save sample counts, selected settings, exclusions, and baseline results.
summary_out = {
    "n_independent_test_pairs": n_pairs_te,
    "n_test_images": n_te,
    "pairs_per_split": n_tab.to_dict(),
    "cnn_variant": FINAL_VARIANT,
    "selected_lr": BEST_LR,
    "hog_C": HOG_C,
    "audit_passed": AUDIT_PASSED,
    "excluded_pairs": pair_status.loc[
        ~pair_status["pair_ok"], "pair_id"
    ].tolist(),
    "manifest_sha256": MAN_SHA,
    "unmodified_balanced_accuracy": {
        m.name: {
            "estimate": get(
                res_img, m.name, "unmodified", "balanced_accuracy"
            )[0],
            "ci_low": get(
                res_img, m.name, "unmodified", "balanced_accuracy"
            )[1],
            "ci_high": get(
                res_img, m.name, "unmodified", "balanced_accuracy"
            )[2]
        }
        for m in MODELS
    }
}
save_json(summary_out, RESULTS / "results_summary.json")


# Include only model files recorded in the plan lock.
saved_lock = json.loads(LOCK_PATH.read_text())
model_files = [
    RESULTS / "models" / name
    for name in saved_lock["model_weight_sha256"]
]

export_files = (
    sorted((RESULTS / "tables").glob("*.csv"))
    + sorted((RESULTS / "tables").glob("*.json"))
    + sorted((RESULTS / "figures").glob("*.png"))
    + model_files
    + [
        LOCK_PATH,
        RESULTS / "environment.json",
        RESULTS / "results_summary.json",
        RESULTS / "split_manifest_sha256.txt",
        MAN_PATH
    ]
)

# Check files before creating the archive.
missing = [str(p) for p in export_files if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Export files missing: {missing}")


# Preserve project-relative paths where possible.
zpath = RESULTS / "results_to_share.zip"

with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in export_files:
        try:
            archive_name = p.resolve().relative_to(ROOT)
        except ValueError:
            archive_name = Path("results") / p.relative_to(RESULTS)

        z.write(p, arcname=archive_name.as_posix())

print(
    "Done. Results archive:", zpath,
    f"({zpath.stat().st_size / 1e6:.1f} MB)"
)
print("Includes tables, figures, and selected models; excludes source TIFFs.")

# Exploratory Extension: Severe Occlusion and Occlusion-Augmented Training

This extension examines performance under stronger simulated occlusion
and whether occlusion-augmented training improves robustness.

Compare the original CNN with a second CNN trained using random white
rectangles. Keep the data split, pretrained backbone, learning rate,
and training seeds unchanged. Select checkpoints using validation data.

Evaluate both models on unmodified images and 50%, 60%, 70%, and 80%
occlusion. This extension was developed after the main test review and
is reported as exploratory.

In [ ]:
# Check that the main notebook has run in this kernel.
required = [
    "use", "canv", "cnn_models", "ResNetHead", "augment",
    "predict_scores", "to_tensor", "box_of", "metrics_basic",
    "Xtr_u8", "Xva_u8", "ytr", "yva", "tr_df", "va_df",
    "BEST_LR", "SEEDS", "RESULTS"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Run the main notebook in this kernel first. Missing: {missing}"
    )

EXT_DIR = RESULTS / "extension"
EXT_DIR.mkdir(parents=True, exist_ok=True)

# Fix settings before examining extension test results.
EXT_LEVELS = (0.0, 0.50, 0.60, 0.70, 0.80)
EXT_DRAWS = 5
OCC_PROB = 0.5
TRAIN_OCC_RANGE = (0.10, 0.70)

# Snapshot the original ensemble.
original_models = list(cnn_models)


def severe_mask(canvas_float, box, frac, rng):
    """Mask approximately frac of the crop with a feasible rectangle."""
    x = np.asarray(canvas_float, dtype=np.float32).copy()
    top, left, bottom, right = box
    h, w = bottom - top, right - left

    if frac == 0:
        return x, {"area_fraction": 0.0, "ink_coverage": 0.0}

    # Relative height × relative width = target area fraction.
    # Both dimensions remain within the crop.
    rel_h = float(np.exp(rng.uniform(np.log(frac), 0.0)))
    rel_w = frac / rel_h
    mh = max(1, min(h, int(round(rel_h * h))))
    mw = max(1, min(w, int(round(rel_w * w))))

    y0 = top + int(rng.integers(0, h - mh + 1))
    x0 = left + int(rng.integers(0, w - mw + 1))

    ink = 1.0 - x
    total = ink[top:bottom, left:right].sum()
    info = {
        "area_fraction": mh * mw / (h * w),
        "ink_coverage": float(
            ink[y0:y0 + mh, x0:x0 + mw].sum() / max(total, 1e-6)
        )
    }

    x[y0:y0 + mh, x0:x0 + mw] = 1.0
    return x, info


def masked_batches(df, frac, draws=EXT_DRAWS):
    """Generate repeatable masks shared by both models."""
    batches, logs = [], []
    K = 1 if frac == 0 else draws

    for k in range(K):
        images = []
        for _, r in df.iterrows():
            rng = np.random.default_rng([
                SEED, zlib.crc32(r.key.encode()),
                int(round(frac * 100)), k, 2026
            ])
            image, info = severe_mask(
                canv[r.key].astype(np.float32) / 255.0,
                box_of(r), frac, rng
            )
            images.append(image)
            logs.append({
                "key": r.key, "pair_id": r.pair_id,
                "target_fraction": frac, "draw": k, **info
            })
        batches.append(np.stack(images))

    return batches, logs


def ensemble_scores(models, images):
    return np.mean([
        predict_scores(model, images) for model in models
    ], axis=0)

In [ ]:
# Explore severity using validation data only.
val_rows = []

for frac in EXT_LEVELS:
    batches, _ = masked_batches(va_df, frac)

    for k, batch in enumerate(batches):
        scores = ensemble_scores(original_models, batch)
        val_rows.append({
            "target_fraction": frac,
            "draw": k,
            **metrics_basic(yva, scores)
        })

val_severity = pd.DataFrame(val_rows)
val_severity.to_csv(
    EXT_DIR / "original_cnn_validation_severity.csv",
    index=False
)

print(
    val_severity.groupby("target_fraction")[
        ["acc", "bal_acc", "logloss"]
    ].mean().round(3).to_string()
)

In [ ]:
def train_occlusion_model(seed):
    """Train a frozen-backbone CNN with additional white occlusion."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    rng = np.random.default_rng(seed + 2026)

    model = ResNetHead(PRETRAINED, unfreeze_layer4=False).to(DEVICE)
    opt = torch.optim.Adam(
        model.head.parameters(),
        lr=BEST_LR,
        weight_decay=WEIGHT_DECAY
    )
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(float(class_ratio)).to(DEVICE)
    )

    images = Xtr_u8.astype(np.float32) / 255.0
    labels = torch.from_numpy(ytr.astype(np.float32))
    boxes = [box_of(r) for _, r in tr_df.iterrows()]

    history = []
    best_loss, best_state, bad = np.inf, None, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        order = torch.randperm(len(images))
        total_loss = 0.0

        for start in range(0, len(images), BATCH):
            idx = order[start:start + BATCH]
            batch = []

            # Apply fresh masks during training.
            for j in idx.tolist():
                image = images[j].copy()
                if rng.random() < OCC_PROB:
                    frac = float(rng.uniform(*TRAIN_OCC_RANGE))
                    image, _ = severe_mask(image, boxes[j], frac, rng)
                batch.append(image)

            xb = augment(to_tensor(np.stack(batch))).to(DEVICE)
            yb = labels[idx].to(DEVICE)

            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(idx)

        # Match the original checkpoint rule: clean validation log-loss.
        scores = predict_scores(model, Xva_u8 / 255.0)
        metrics = metrics_basic(yva, scores)
        history.append({
            "seed": seed,
            "epoch": epoch + 1,
            "train_loss": total_loss / len(images),
            "val_loss": metrics["logloss"],
            "val_bal_acc": metrics["bal_acc"]
        })

        if metrics["logloss"] < best_loss - 1e-4:
            best_loss = metrics["logloss"]
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.head.state_dict().items()
            }
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.head.load_state_dict(best_state)
    torch.save(best_state, EXT_DIR / f"occlusion_head_seed{seed}.pt")
    print(f"seed {seed}: best validation loss {best_loss:.4f}")
    return model, pd.DataFrame(history)


# Keep the original models unchanged.
occlusion_models, extension_histories = [], []

for sd in SEEDS:
    model, history = train_occlusion_model(sd)
    occlusion_models.append(model)
    extension_histories.append(history)

pd.concat(extension_histories, ignore_index=True).to_csv(
    EXT_DIR / "occlusion_training_history.csv",
    index=False
)

In [ ]:
val_comparison = []

for frac in EXT_LEVELS:
    batches, _ = masked_batches(va_df, frac)

    for name, models in [
        ("original", original_models),
        ("occlusion_augmented", occlusion_models)
    ]:
        for k, batch in enumerate(batches):
            scores = ensemble_scores(models, batch)
            val_comparison.append({
                "model": name,
                "target_fraction": frac,
                "draw": k,
                **metrics_basic(yva, scores)
            })

val_comparison = pd.DataFrame(val_comparison)
val_comparison.to_csv(
    EXT_DIR / "validation_comparison.csv", index=False
)

print(
    val_comparison.groupby(["model", "target_fraction"])[
        ["acc", "bal_acc", "logloss"]
    ].mean().round(3).to_string()
)

In [ ]:
# Record the extension settings before generating test predictions.
save_json({
    "analysis": "post-hoc exploratory extension",
    "levels": EXT_LEVELS,
    "draws": EXT_DRAWS,
    "training_occlusion_probability": OCC_PROB,
    "training_occlusion_range": TRAIN_OCC_RANGE,
    "learning_rate": BEST_LR,
    "seeds": list(SEEDS),
    "checkpoint_rule": "unmodified validation log-loss",
    "mask_rule": "log-uniform crop-relative height; width = fraction / height"
}, EXT_DIR / "extension_plan.json")

test_rows, delta_rows, prediction_rows, mask_logs = [], [], [], []

for frac in EXT_LEVELS:
    batches, logs = masked_batches(te_df, frac)
    mask_logs.extend(logs)
    boot_by_model, point_by_model = {}, {}

    for name, models in [
        ("original", original_models),
        ("occlusion_augmented", occlusion_models)
    ]:
        scores = np.stack([
            ensemble_scores(models, batch) for batch in batches
        ])
        yh = (scores > 0.5).astype(int)

        point = boot_metrics(yte, yh, ones_i)
        boot = boot_metrics(yte, yh, Wi)
        point_by_model[name] = point["balanced_accuracy"].mean()
        boot_by_model[name] = boot["balanced_accuracy"].mean(axis=1)
        lo, hi = np.percentile(boot_by_model[name], CI)

        test_rows.append({
            "model": name,
            "target_fraction": frac,
            "balanced_accuracy": float(point_by_model[name]),
            "ci_low": lo,
            "ci_high": hi
        })

        for k in range(len(batches)):
            prediction_rows.append(pd.DataFrame({
                "model": name,
                "target_fraction": frac,
                "draw": k,
                "key": te_df["key"].to_numpy(),
                "pair_id": te_df["pair_id"].to_numpy(),
                "y_true": yte,
                "score_nike": scores[k]
            }))

    # Positive differences favour occlusion-augmented training.
    difference = (
        boot_by_model["occlusion_augmented"]
        - boot_by_model["original"]
    )
    lo, hi = np.percentile(difference, CI)
    delta_rows.append({
        "target_fraction": frac,
        "delta_balanced_accuracy": float(
            point_by_model["occlusion_augmented"]
            - point_by_model["original"]
        ),
        "ci_low": lo,
        "ci_high": hi
    })

ext_metrics = pd.DataFrame(test_rows)
ext_deltas = pd.DataFrame(delta_rows)

ext_metrics.to_csv(EXT_DIR / "test_metrics.csv", index=False)
ext_deltas.to_csv(EXT_DIR / "paired_model_differences.csv", index=False)
pd.concat(prediction_rows, ignore_index=True).to_csv(
    EXT_DIR / "test_predictions.csv", index=False
)
pd.DataFrame(mask_logs).to_csv(
    EXT_DIR / "test_mask_coverage.csv", index=False
)

print("\nExploratory image-level test results:")
print(ext_metrics.round(3).to_string(index=False))
print("\nAugmented minus original balanced accuracy:")
print(ext_deltas.round(3).to_string(index=False))